# SparkCaster Time Series Forecasting System

SparkCaster gateway: `http://spark-gateway.kubeflow.svc.cluster.local:8888`

## Overview
A **distributed time series forecasting system on SparkCaster**. Each
(metric × grouping × group) series is one task; models are fit on Spark
executors and the best model per series is selected automatically.

- **Cluster:** ~20 executors × 4 cores (dynamic allocation 5–25), 16 GB/executor
- **Data source:** Nessie catalog table `sandbox.sandbox_finance.dcgm_metrics_summary_imputed`
- **Data stays in Spark** until results are collected (no early `.toPandas()`)
- **Python 3.11**

## Bootstrap (driver vs executor are intentionally separate)
A single dependency manifest `PACKAGES = {"driver": [...], "executor": [...]}`
drives three distinct install paths:
1. **Driver setup** (`bootstrap_driver`) — interactive kernel, has internet, `pip install --user`; full analysis/vis stack.
2. **Executor prep** — a **wheelhouse** zip of the minimal executor libs, built on the driver and shipped via `SparkFiles`.
3. **Executor runtime** — each task installs from that wheelhouse **offline** (workers have no internet).

## Models (5, fit per series on executors)
1. **Exponential Smoothing** (additive trend + seasonal)
2. **ARIMA** `(5,1,0)`
3. **SARIMA** `(1,1,1)(1,1,1,7)` (weekly seasonality)
4. **Prophet** (native 80% intervals)
5. **Holt-Winters** (damped, multiplicative seasonal)

Best model chosen by lowest MAE. **P10/P50/P90 prediction intervals** and
**`[0,1]` bounds for utilization metrics** are computed on the executors, so
the distributed output matches what plotting/export consume (one canonical
result schema, documented in `forecast_time_series_row`).

- **Train/test split:** `TRAIN_SPLIT = 0.7`  ·  **Horizon:** `FORECAST_DAYS = 1100` (~3 years)
- These are the single source of truth; `CONFIG` mirrors them.

## Metrics forecasted (8)
`gpu_util_p50`, `tensor_util_p50/p95/p99`, `chip_power_p50/p95`,
`redfish_power_p50/p95`.
*(tflops_* metrics are excluded — not present in this table.)*

## Grouping strategies (8)
`All`, `product_resolved`, `product_segment`, `customer_segment`,
`region_summary`, `region_summary+product_segment`,
`region_summary+product_resolved`, `product_segment+product_resolved`.

## Outputs → CoreWeave Object Storage (LOTA)
Final pandas frames are exported as **CSV + XLSX** to
`s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/<timestamp>/`
(one run-scoped timestamp), then downloaded from Cloud Console. Datasets:

| Object (`.csv` and `.xlsx`) | Contents |
|---|---|
| `all_models_results` | Metrics for every model × series |
| `best_models_results` | Best model per series |
| `forecast_daily_values` | Daily P10/P50/P90 forecast (best model) |
| `forecast_monthly_values` | Monthly aggregate (best model) |
| `forecast_monthly_values_all` | Monthly aggregate (all models) |
| `forecast_monthly_values_all_with_history` | Monthly actuals + forecast (all models) |

## How to run
**Restart & Run All**, top to bottom. You will be prompted (via `getpass`) for:
- the encrypted-keyring master password (once), then CAIOS credentials (for Nessie);
- your CoreWeave Object Storage **access key / secret** at the export helper
  (or set `CW_S3_ACCESS_KEY` / `CW_S3_SECRET_KEY` in the kernel to skip the prompt).

## Architecture
1. `create_time_series_tasks` builds one Spark row per (metric, grouping, group_key) with its `(day, value)` array.
2. `forecast_time_series_row` runs on each executor (offline dependency install → fit 5 models → intervals + bounds → canonical JSON).
3. `run_sparkcaster_forecasting` distributes via RDD `map` and collects results to the driver.
4. Results are expanded to pandas, aggregated to daily/monthly grains, and exported to object storage.


In [1]:
# ── SETUP: logging helpers + dependency manifest + DRIVER bootstrap ──────────
import importlib, subprocess, sys, os, site

# --- Logging helpers (used across the notebook instead of ad-hoc print blocks) ---
def log(*args, **kwargs):
    """Drop-in for print(); central hook for future structured logging."""
    print(*args, **kwargs)

def log_section(title, char="=", width=80):
    """Banner header: rule / title / rule."""
    print(char * width)
    print(title)
    print(char * width)

# --- Single dependency manifest, shared by every install path ----------------
# WHY driver vs executor differ:
#   * DRIVER (this kernel): interactive, has internet -> pip install --user.
#     Installs the full analysis/vis stack used only on the driver.
#   * EXECUTOR (Spark workers): no internet -> installed OFFLINE from a
#     wheelhouse zip (see the wheelhouse cell) at task runtime (see the
#     forecasting cell). Only the minimal libs each task needs to fit models.
PACKAGES = {
    "driver": [
        ("keyring", "keyring"),
        ("ipython-secrets", "ipython_secrets"),
        ("oauth2client", "oauth2client"),
        ("pyarrow", "pyarrow"),
        ("fsspec", "fsspec"),
        ("s3fs", "s3fs"),
        ("scipy", "scipy"),
        ("statsmodels", "statsmodels"),
        ("matplotlib", "matplotlib"),
        ("scikit-learn", "sklearn"),
        ("keyrings.cryptfile", "keyrings"),   # plugin under the 'keyrings' pkg
        ("bokeh==3.6.2", "bokeh"),
        ("jupyter_bokeh", "jupyter_bokeh"),
        ("panel==1.5.2", "panel"),
        ("holoviews==1.19.0", "holoviews"),
        ("hvplot==0.10.0", "hvplot"),
        ("datashader==0.16.3", "datashader"),
        ("dask[dataframe]==2024.9.1", "dask"),
        ("distributed==2024.9.1", "distributed"),
        ("reportlab", "reportlab"),
        ("prophet", "prophet"),
        ("openpyxl", "openpyxl"),
        ("tqdm", "tqdm"),
    ],
    # Minimal libs each executor needs to fit models; downloaded to the
    # wheelhouse on the driver and pip-installed offline on the workers.
    "executor": ["statsmodels", "scipy", "pandas", "numpy", "patsy"],
}

def ensure_user_site():
    """Make user site-packages / user bin visible in this kernel."""
    user_site = site.getusersitepackages()
    if user_site and user_site not in sys.path:
        sys.path.insert(0, user_site)
    user_bin = os.path.expanduser("~/.local/bin")
    if user_bin not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{user_bin}:{os.environ.get('PATH','')}"
    return user_site, user_bin

def is_module_available(module_name):
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

def install_if_missing(pip_name, import_name=None):
    """DRIVER install: pip install --user only if the import is missing."""
    import_name = import_name or pip_name
    if not is_module_available(import_name):
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", pip_name])
    else:
        print(f"{pip_name} already installed.")

def bootstrap_driver():
    """Install the full driver-side stack from the manifest (one driver path)."""
    _user_site, _user_bin = ensure_user_site()
    log(f"User site-packages: {_user_site}")
    log(f"User bin: {_user_bin}")
    for pip_name, import_name in PACKAGES["driver"]:
        install_if_missing(pip_name, import_name)
    log("✓ All driver packages installed/verified")

log_section("DRIVER BOOTSTRAP")
bootstrap_driver()


DRIVER BOOTSTRAP
User site-packages: /home/spark/.local/lib/python3.10/site-packages
User bin: /home/spark/.local/bin
keyring already installed.
ipython-secrets already installed.
Installing oauth2client ...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [oauth2client]
pyarrow already installed.
fsspec already installed.
s3fs already installed.
scipy already installed.
Installing statsmodels ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 41.1 MB/s  0:00:00ta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]
Installing matplotlib ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 37.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 188.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]
scikit-learn already installed.
keyrings.cryptfile already installed.
Installing bokeh==3.6.2 ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 31.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bokeh]32m1/2 [bokeh]
Installing jupyter_bokeh ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# SPARKCASTER DISTRIBUTED PROCESSING
# Using SparkCaster for distributed execution across the cluster
# No multiprocessing needed - Spark handles task distribution

print("✓ Using SparkCaster for distributed processing")
print("  Tasks will be distributed across all Spark executors")
print("  No single-node memory limits or CPU constraints")

✓ Using SparkCaster for distributed processing
  Tasks will be distributed across all Spark executors
  No single-node memory limits or CPU constraints


In [3]:
# ── Credentials: initialize the encrypted keyring (shared by CAIOS) ──────────
import keyring, os
from getpass import getpass
from keyrings.cryptfile.cryptfile import CryptFileKeyring
from pathlib import Path

_KEYRING_READY = False

def ensure_keyring():
    """Initialize the CryptFile keyring exactly once per kernel (idempotent).

    The CAIOS credentials cell calls this; the master password is only
    prompted the first time.
    """
    global _KEYRING_READY
    if _KEYRING_READY:
        return
    os.environ["KEYRING_CRYPTFILE_PATH"] = f"{Path.home()}/.local/share/python_keyring/cryptfile_pass.cfg"
    kr = CryptFileKeyring()
    kr.keyring_key = getpass("Set/enter master password for encrypted keyring: ")
    keyring.set_keyring(kr)
    _KEYRING_READY = True

ensure_keyring()


In [4]:
# Common imports
import pandas as pd
import numpy as np


In [5]:
# ── CAIOS credentials (reuses the keyring initialized above) ────────────────
ensure_keyring()   # no-op / no re-prompt if already initialized this kernel

caios_access_key = keyring.get_password("caios", "access_key")
caios_secret_key = keyring.get_password("caios", "secret_key")
if not caios_access_key:
    caios_access_key = input("Enter CAIOS access key: ")
    keyring.set_password("caios", "access_key", caios_access_key)
if not caios_secret_key:
    caios_secret_key = getpass("Enter CAIOS secret key: ")
    keyring.set_password("caios", "secret_key", caios_secret_key)

log("✓ CAIOS credentials configured")


✓ CAIOS credentials configured


In [6]:
#pull in data with CLUSTER resources (not local mode!)
# 
from spark.nessie import NessieSparkClient
from pyspark.sql import SparkSession
import sys
import site
import os

# Get user site-packages path
user_site = site.getusersitepackages()
print(f"User site-packages: {user_site}")

# Configure Spark to use cluster resources AND install packages on executors
spark = SparkSession.builder \
    .appName("NessieTimeSeriesForecast") \
    .config("spark.executor.memory", "16g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.instances", "20") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "10000") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "5") \
    .config("spark.dynamicAllocation.maxExecutors", "25") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.pyspark.python", sys.executable) \
    .config("spark.pyspark.driver.python", sys.executable) \
    .getOrCreate()

print("="*60)
print("SPARK CLUSTER CONFIGURATION")
print("="*60)
print(f"Executor Memory: {spark.conf.get('spark.executor.memory')}")
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"Executor Cores: {spark.conf.get('spark.executor.cores')}")
print(f"Executor Instances: {spark.conf.get('spark.executor.instances')}")
print(f"Dynamic Allocation: {spark.conf.get('spark.dynamicAllocation.enabled')}")
print(f"Min/Max Executors: {spark.conf.get('spark.dynamicAllocation.minExecutors')}/{spark.conf.get('spark.dynamicAllocation.maxExecutors')}")
print("="*60)

# Executor dependencies are handled elsewhere, NOT here:
#   * wheelhouse build cell -> downloads the offline wheels
#   * forecasting cell -> installs them on each executor at task runtime
# (Driver-side online installs must not be pushed to workers, which have no internet.)
    
# Set up Nessie Spark client
ness = NessieSparkClient(
    svc_url="http://kf-proxy.nessie.svc.cluster.local:19120/api/v2",
    nessie_endpoint="http://nessie-prd.cwobject.com",
    caios_access_key=caios_access_key,
    caios_secret_key=caios_secret_key,
    dbtcaster=True,
)
# Turn off warnings
spark.sparkContext.setLogLevel("ERROR")


User site-packages: /home/spark/.local/lib/python3.10/site-packages
SPARK CLUSTER CONFIGURATION
Executor Memory: 16g
Driver Memory: 8g
Executor Cores: 4
Executor Instances: 20
Dynamic Allocation: true
Min/Max Executors: 5/25


In [7]:

# Load data from Nessie catalog
df = ness.sql("select * from sandbox.sandbox_finance.dcgm_metrics_summary_imputed_daily_v2")
df.show(5, truncate=False)
print(f"\nTotal rows: {df.count():,}")

+-------------------+----------+-------------+-----------+---------+-----------+------------+----------------+---------------+------------------+----------------+-------------+---------------+--------------------+-------------------+-------------------+-------------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+------------+------------+------------+-------------------+-------------------+-------------------+--------------------------+--------------------------+--------------------------+------------------------------+------------------------------+------------------------------+---------------------------+---------------------------+---------------------------+--------------------------+--------------------------+--------------------------+-----------------------+-----------------------+----------------------

In [8]:
# BUILD WHEELHOUSE FOR EXECUTORS (no internet needed on workers)
import os, sys, subprocess, shutil
from pyspark import SparkFiles

wheel_dir = '/tmp/sparkcaster_wheels'
zip_base = '/tmp/sparkcaster_wheels'
zip_path = f'{zip_base}.zip'

if not os.path.exists(zip_path):
    os.makedirs(wheel_dir, exist_ok=True)
    # Download wheels on driver (executors will install from this zip)
    packages = PACKAGES['executor']   # single manifest, shared with the executor runtime install
    subprocess.check_call([sys.executable, '-m', 'pip', 'download', '-d', wheel_dir] + packages)
    shutil.make_archive(zip_base, 'zip', wheel_dir)

spark.sparkContext.addFile(zip_path)
print(f'✓ Distributed wheelhouse: {zip_path}')


  Using cached statsmodels-0.14.6-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (9.5 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (10.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 55.2 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 165.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 161.0 MB/s  0:00:00
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)
Saved /tmp/sparkcaster_wheels/statsmodels-0.14.6-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
Saved /tmp/sparkcaster_wheels/numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Saved /tmp/sparkcaster_wheels/scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Saved /tmp/sparkcaster_wheels/pandas-2.3.

In [9]:
# Keep data as Spark DataFrame for distributed processing
from pyspark.sql.functions import col, when, to_date
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Preparing Spark DataFrame for distributed forecasting...")

# Convert day column to date type (if not already)
df = df.withColumn('day', to_date(col('day')))

# Create region_summary field using Spark operations
# Logic: if region starts with 'EU' then 'EU', else 'NAM'
df = df.withColumn('region_summary', 
                   when(col('region').startswith('EU'), 'EU')
                   .otherwise('NAM'))

# Cache the DataFrame for faster access during distributed processing
df = df.cache()

# Get basic statistics (collect only summary info, not full data)
row_count = df.count()
date_range = df.select(F.min('day'), F.max('day')).first()
columns = df.columns

print(f"Data shape: {row_count} rows × {len(columns)} columns")
print(f"Date range: {date_range[0]} to {date_range[1]}")
print(f"\nColumns: {columns}")
print(f"\nRegion Summary Distribution:")
df.groupBy('region_summary').count().show()
print(f"\nOriginal Regions → Region Summary Mapping:")
df.select('region', 'region_summary').distinct().orderBy('region').show()
print(f"\nSample data (first 5 rows):")
df.show(5)

print(f"\n✓ Data ready for SparkCaster distributed forecasting")
print(f"  DataFrame cached with {row_count:,} rows")

Preparing Spark DataFrame for distributed forecasting...
Data shape: 119548 rows × 82 columns
Date range: 2025-01-23 to 2026-06-23

Columns: ['day', 'region', 'region_rollup', 'zone', 'gen', 'is_training', 'is_multinode', 'product_resolved', 'product_segment', 'gpu_count_expected', 'customer_segment', 'customer_name', 'peak_power_unit', 'gpu_util_p50', 'gpu_util_p95', 'gpu_util_p99', 'tensor_util_p50', 'tensor_util_p95', 'tensor_util_p99', 'dram_active_p50', 'dram_active_p95', 'dram_active_p99', 'mem_copy_util_p50', 'mem_copy_util_p95', 'mem_copy_util_p99', 'sm_active_p50', 'sm_active_p95', 'sm_active_p99', 'sm_clock_p50', 'sm_clock_p95', 'sm_clock_p99', 'sm_occupancy_p50', 'sm_occupancy_p95', 'sm_occupancy_p99', 'memory_boundness_index_p50', 'memory_boundness_index_p95', 'memory_boundness_index_p99', 'TFLOPS_per_watt_efficiency_p50', 'TFLOPS_per_watt_efficiency_p95', 'TFLOPS_per_watt_efficiency_p99', 'compute_occupancy_index_p50', 'compute_occupancy_index_p95', 'compute_occupancy_inde

In [48]:
# Define metrics to forecast
# Note: Using actual column names from dcgm_metrics_summary_imputed table
METRICS = [
    'gpu_util_p50',
    'gpu_util_p95',
    'tensor_util_p50', 
    'tensor_util_p95', 
    'chip_power_fleet_p50', 
    'chip_power_fleet_p95',
    'redfish_power_fleet_p50', 
    'redfish_power_fleet_p95',
    'tensor_tflops_fleet_p50',
    'tensor_tflops_fleet_p95'
    # Note: tflops columns are named differently in this table
    # 'tflops_total_p50',  # doesn't exist - use tflops_p50
    # 'tflops_total_p95',  # doesn't exist - use tflops_p95
    # 'tflops_node_avg_p50',  # need to verify actual column name
    # 'tflops_node_avg_p95',  # need to verify actual column name
    # 'tflops_node_avg_p99'   # need to verify actual column name
]

print(f"⚠️  WARNING: Some tflops metrics commented out - need to verify column names")
print(f"   Available tflops columns in error message: tflops_p50, tflops_p95")
print(f"   You may want to check df.columns to see all available metrics\n")

# Define grouping columns (using _norm suffix for normalized columns)
# Using region_summary instead of region (EU vs NAM)
GROUPINGS = {
    'All': [],
    'product_resolved': ['product_resolved'],
    'product_segment': ['product_segment'],
    'customer_segment': ['customer_segment'],
    'region_summary': ['region_rollup'],
    'region_summary+product_segment': ['region_rollup', 'product_segment'],  # Combined grouping
    'region_summary+product_resolved': ['region_rollup', 'product_resolved'],  # Combined grouping
    'product_segment+product_resolved': ['product_segment', 'product_resolved']
}

print(f"Metrics to forecast: {len(METRICS)}")
print(f"Grouping strategies: {list(GROUPINGS.keys())}")
print(f"\nGrouping details:")
print(f"  - All: Global aggregate")
print(f"  - product_resolved: By GPU type (H100, H200, L40, etc.)")
print(f"  - product_segment: By segment (HGX, PCIE)")
print(f"  - customer_segment: By customer type")
print(f"  - region_summary: By region (EU vs NAM)")
print(f"  - region_summary+product_segment: By region AND segment (e.g., EU-HGX, NAM-PCIE)")
print(f"  - region_summary+product_resolved: By region AND product (e.g., EU-B200, NAM-GB200)")
print(f"\nTotal combinations: {len(METRICS)} metrics × {len(GROUPINGS)} groupings = {len(METRICS) * len(GROUPINGS)} series")

print(f"\n💡 TIP: Run df.columns to see all available column names if you want to add more metrics")

⚠️  WARNING: Some tflops metrics commented out - need to verify column names
   Available tflops columns in error message: tflops_p50, tflops_p95
   You may want to check df.columns to see all available metrics

Metrics to forecast: 10
Grouping strategies: ['All', 'product_resolved', 'product_segment', 'customer_segment', 'region_summary', 'region_summary+product_segment', 'region_summary+product_resolved', 'product_segment+product_resolved']

Grouping details:
  - All: Global aggregate
  - product_resolved: By GPU type (H100, H200, L40, etc.)
  - product_segment: By segment (HGX, PCIE)
  - customer_segment: By customer type
  - region_summary: By region (EU vs NAM)
  - region_summary+product_segment: By region AND segment (e.g., EU-HGX, NAM-PCIE)
  - region_summary+product_resolved: By region AND product (e.g., EU-B200, NAM-GB200)

Total combinations: 10 metrics × 8 groupings = 80 series

💡 TIP: Run df.columns to see all available column names if you want to add more metrics


In [49]:
# Data preprocessing - Create Spark DataFrame with all metric/grouping combinations
from pyspark.sql.functions import col, concat_ws, avg, collect_list, struct
from pyspark.sql import Window

print("Preparing time series combinations for distributed processing...")

# We'll create a DataFrame where each row represents a metric/grouping/group_key combination
# This will be partitioned and distributed to executors via SparkCaster

def create_time_series_tasks(spark_df, metrics, groupings):
    """
    Create a Spark DataFrame where each row represents one time series forecasting task
    
    Returns: Spark DataFrame with columns:
    - metric: the metric name
    - grouping_name: the grouping strategy name  
    - group_key: the specific group identifier
    - time_series_data: array of structs with (day, value)
    """
    from pyspark.sql.functions import lit, collect_list, struct, concat_ws
    
    tasks = []
    
    # For each metric and grouping combination
    for metric in metrics:
        for grouping_name, group_cols in groupings.items():
            
            if len(group_cols) == 0:
                # 'All' grouping - aggregate everything by day
                agg_df = spark_df.groupBy('day').agg(
                    avg(metric).alias('value')
                ).withColumn('group_key', lit('All'))
                
            else:
                # Group by specified columns + day
                agg_df = spark_df.groupBy(*(group_cols + ['day'])).agg(
                    avg(metric).alias('value')
                ).withColumn('group_key', concat_ws('_', *group_cols))
            
            # Add metadata columns and select ONLY the columns we need (consistent schema)
            agg_df = agg_df.select(
                lit(metric).alias('metric'),
                lit(grouping_name).alias('grouping_name'),
                col('group_key'),
                col('day'),
                col('value')
            )
            
            tasks.append(agg_df)
    
    # Union all tasks - now they all have the same 5 columns
    from functools import reduce
    all_tasks = reduce(lambda df1, df2: df1.union(df2), tasks)
    
    # Group by metric/grouping/group_key to create array of (day, value) pairs
    result = all_tasks.groupBy('metric', 'grouping_name', 'group_key').agg(
        collect_list(struct('day', 'value')).alias('time_series_data')
    )
    
    return result

print("✓ Data preparation function created")
print("  This will create one task per metric/grouping/group_key combination")

Preparing time series combinations for distributed processing...
✓ Data preparation function created
  This will create one task per metric/grouping/group_key combination


In [ ]:
# ── EXECUTOR FORECASTING (runs on Spark workers) ─────────────────────────────
# One task per (metric, grouping, group_key). Fits 5 models, picks the best by
# MAE, and returns the CANONICAL result schema (documented in the function).
# Prediction intervals (P10/P90) and [0,1] utilization bounds are computed HERE
# so the distributed path produces exactly what plotting/export consume.
import time, json
import pandas as pd
import numpy as np

# --- Single source of truth for split / horizon (CONFIG mirrors these) -------
TRAIN_SPLIT = 0.7
FORECAST_END_DATE = pd.Timestamp("2030-12-31")
FORECAST_DAYS = (FORECAST_END_DATE - pd.Timestamp(df.select(F.max("day")).first()[0])).days
_Z_P10_P90 = 1.2816            # 10th/90th percentile of a normal (80% central band)
EXECUTOR_PACKAGES = PACKAGES["executor"]   # from the dependency manifest (cell 1)


def forecast_time_series_row(row):
    """Fit models for one time series on an executor; return a JSON result.

    CANONICAL RESULT SCHEMA (the single contract shared by forecasting,
    plotting, and export):
      { metric, grouping, grouping_name, group_key,
        status: 'completed'|'skipped'|'error',
        best_model, train_size, test_size,
        train_dates[], test_dates[], train_values[], test_values[],
        results: { <model>: {
            status: 'success'|'failed',
            metrics: {MSE, RMSE, MAPE, MAE},
            mae,                    # convenience copy for best-model ranking
            test_predictions[],     # aligned to test window
            forecast[],             # P50 point forecast, length FORECAST_DAYS
            forecast_lower[],       # P10
            forecast_upper[],       # P90
            fitted[],               # in-sample fit, aligned to train (may be [])
        } } }
    """
    import subprocess, sys, warnings, importlib, os
    warnings.filterwarnings('ignore')

    # --- Executor runtime dependency ensure: OFFLINE install from wheelhouse ---
    # WHY separate from the driver: workers have no internet, so we install the
    # minimal manifest from the wheelhouse zip shipped via SparkFiles.
    need = []
    for mod in ('statsmodels', 'scipy'):
        try:
            importlib.import_module(mod)
        except ImportError:
            need.append(mod)
    if need:
        try:
            import zipfile, tempfile
            from pyspark import SparkFiles
            wheel_zip = SparkFiles.get('sparkcaster_wheels.zip')
            extract_dir = os.path.join(tempfile.gettempdir(), 'sparkcaster_wheels')
            if not os.path.exists(extract_dir):
                os.makedirs(extract_dir, exist_ok=True)
                with zipfile.ZipFile(wheel_zip, 'r') as zf:
                    zf.extractall(extract_dir)
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', '--user', '--no-index',
                 '--find-links', extract_dir] + list(EXECUTOR_PACKAGES),
                stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL, timeout=180)
            importlib.reload(importlib.import_module('site'))
        except Exception as install_error:
            return json.dumps({
                'metric': getattr(row, 'metric', 'unknown'),
                'grouping': getattr(row, 'grouping_name', 'unknown'),
                'grouping_name': getattr(row, 'grouping_name', 'unknown'),
                'group_key': getattr(row, 'group_key', 'unknown'),
                'status': 'error',
                'error': f'Executor package install failed: {str(install_error)[:150]}',
            })

    import numpy as np
    import pandas as pd

    metric_name = getattr(row, 'metric', 'unknown')
    grouping_name = getattr(row, 'grouping_name', 'unknown')
    group_key = getattr(row, 'group_key', 'unknown')

    def _err(msg):
        return json.dumps({'metric': metric_name, 'grouping': grouping_name,
                           'grouping_name': grouping_name, 'group_key': group_key,
                           'status': 'error', 'error': str(msg)[:200]})

    def _skip(reason, n):
        return json.dumps({'metric': metric_name, 'grouping': grouping_name,
                           'grouping_name': grouping_name, 'group_key': group_key,
                           'status': 'skipped', 'reason': reason, 'data_points': n})

    try:
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        from statsmodels.tsa.arima.model import ARIMA
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        try:
            from prophet import Prophet
            has_prophet = True
        except Exception:
            has_prophet = False

        ts = pd.DataFrame([
            {'day': pd.to_datetime(p.day),
             'value': float(p.value) if p.value is not None else 0.0}
            for p in row.time_series_data
        ])
        if len(ts) < 21:
            return _skip('insufficient_data', len(ts))
        ts = ts.dropna().sort_values('day').reset_index(drop=True)
        if len(ts) < 21:
            return _skip('insufficient_data_after_cleaning', len(ts))

        split_idx = int(len(ts) * TRAIN_SPLIT)
        train = ts.iloc[:split_idx].copy()
        test = ts.iloc[split_idx:].copy()
        train_vals = train['value'].to_numpy(dtype=float)
        test_vals = test['value'].to_numpy(dtype=float)
        n_test = len(test_vals)
        horizon = n_test + FORECAST_DAYS

        util = any(k in metric_name.lower()
                   for k in ('util', 'utilization', 'usage', 'saturation'))

        def _clip(a):
            a = np.asarray(a, dtype=float)
            a = np.maximum(a, 0.0)               # metrics are non-negative
            if util:
                a = np.minimum(a, 1.0)           # utilization bounded to [0, 1]
            return a

        def _metrics(actual, pred):
            actual = np.asarray(actual, float); pred = np.asarray(pred, float)
            mse = float(np.mean((actual - pred) ** 2))
            rmse = float(np.sqrt(mse))
            denom = np.where(actual == 0, np.nan, actual)
            mape = float(np.nanmean(np.abs((actual - pred) / denom)) * 100)
            mae = float(np.mean(np.abs(actual - pred)))
            return {'MSE': mse, 'RMSE': rmse, 'MAPE': mape, 'MAE': mae}

        def _pack(full_forecast, fitted_full):
            """Split a length-`horizon` forecast into test + future and build the
            canonical per-model dict. Intervals are an empirical residual band."""
            full_forecast = np.asarray(full_forecast, float)
            test_pred = full_forecast[:n_test]
            future = full_forecast[n_test:n_test + FORECAST_DAYS]
            if fitted_full is not None and len(fitted_full) == len(train_vals):
                resid = train_vals - np.asarray(fitted_full, float)
            else:
                resid = test_vals - test_pred[:len(test_vals)]
            sigma = float(np.nanstd(resid)) if len(resid) else 0.0
            band = _Z_P10_P90 * sigma
            m = _metrics(test_vals, test_pred)
            return {
                'status': 'success',
                'metrics': m,
                'mae': m['MAE'],
                'test_predictions': _clip(test_pred).tolist(),
                'forecast': _clip(future).tolist(),
                'forecast_lower': _clip(future - band).tolist(),
                'forecast_upper': _clip(future + band).tolist(),
                'fitted': (_clip(fitted_full).tolist() if fitted_full is not None else []),
            }

        results = {}

        # Model 1: Exponential Smoothing (additive trend + seasonal)
        try:
            fit = ExponentialSmoothing(train_vals, seasonal_periods=7,
                                       trend='add', seasonal='add').fit()
            fc = np.asarray(fit.forecast(steps=horizon), float)
            results['exponential_smoothing'] = _pack(fc, getattr(fit, 'fittedvalues', None))
        except Exception as e:
            results['exponential_smoothing'] = {'status': 'failed', 'error': str(e)[:100]}

        # Model 2: ARIMA
        try:
            fit = ARIMA(train_vals, order=(5, 1, 0)).fit()
            fc = np.asarray(fit.forecast(steps=horizon), float)
            results['arima'] = _pack(fc, getattr(fit, 'fittedvalues', None))
        except Exception as e:
            results['arima'] = {'status': 'failed', 'error': str(e)[:100]}

        # Model 3: SARIMA (weekly seasonality)
        try:
            fit = SARIMAX(train_vals, order=(1, 1, 1),
                          seasonal_order=(1, 1, 1, 7)).fit(disp=False)
            fc = np.asarray(fit.forecast(steps=horizon), float)
            results['sarima'] = _pack(fc, getattr(fit, 'fittedvalues', None))
        except Exception as e:
            results['sarima'] = {'status': 'failed', 'error': str(e)[:100]}

        # Model 4: Prophet (uses its NATIVE 80% intervals when available)
        if has_prophet:
            try:
                pdf = train.rename(columns={'day': 'ds', 'value': 'y'})
                m = Prophet(daily_seasonality=True, weekly_seasonality=True,
                            yearly_seasonality=False, interval_width=0.8)
                m.fit(pdf)
                fdf = m.make_future_dataframe(periods=horizon)
                pred = m.predict(fdf)
                yhat = pred['yhat'].to_numpy()
                lo = pred['yhat_lower'].to_numpy()
                hi = pred['yhat_upper'].to_numpy()
                ntr = len(train_vals)
                fitted_full = yhat[:ntr]
                test_pred = yhat[ntr:ntr + n_test]
                future = yhat[ntr + n_test:ntr + horizon]
                low = lo[ntr + n_test:ntr + horizon]
                up = hi[ntr + n_test:ntr + horizon]
                mm = _metrics(test_vals, test_pred)
                results['prophet'] = {
                    'status': 'success', 'metrics': mm, 'mae': mm['MAE'],
                    'test_predictions': _clip(test_pred).tolist(),
                    'forecast': _clip(future).tolist(),
                    'forecast_lower': _clip(low).tolist(),
                    'forecast_upper': _clip(up).tolist(),
                    'fitted': _clip(fitted_full).tolist(),
                }
            except Exception as e:
                results['prophet'] = {'status': 'failed', 'error': str(e)[:100]}

        # Model 5: Holt-Winters (damped, multiplicative seasonal)
        try:
            fit = ExponentialSmoothing(train_vals, seasonal_periods=7, trend='add',
                                       seasonal='mul', damped_trend=True).fit()
            fc = np.asarray(fit.forecast(steps=horizon), float)
            results['holt_winters'] = _pack(fc, getattr(fit, 'fittedvalues', None))
        except Exception as e:
            results['holt_winters'] = {'status': 'failed', 'error': str(e)[:100]}

        ok = {k: v for k, v in results.items() if v.get('status') == 'success'}
        best_model = min(ok.items(), key=lambda kv: kv[1]['mae'])[0] if ok else None

        return json.dumps({
            'metric': metric_name, 'grouping': grouping_name, 'grouping_name': grouping_name,
            'group_key': group_key, 'status': 'completed', 'best_model': best_model,
            'results': results, 'train_size': len(train), 'test_size': len(test),
            'train_dates': train['day'].astype(str).tolist(),
            'test_dates': test['day'].astype(str).tolist(),
            'train_values': train['value'].tolist(),
            'test_values': test['value'].tolist(),
        })

    except ImportError as ie:
        return _err(f'Import failed after install: {ie}')
    except Exception as e:
        return _err(e)


print("✓ Executor forecasting defined (P10/P90 intervals + [0,1] util bounds, canonical schema)")


✓ Executor forecasting defined (P10/P90 intervals + [0,1] util bounds, canonical schema)


In [51]:
# SPARK DISTRIBUTED ORCHESTRATION - Run distributed forecasting
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
import time
import json

def run_sparkcaster_forecasting(spark_df, metrics, groupings):
    """
    Run time series forecasting using Spark distributed processing

    Parameters:
    - spark_df: Spark DataFrame with time series data
    - metrics: List of metrics to forecast
    - groupings: Dictionary of grouping strategies

    Returns: List of dict results
    """

    print(f"{'='*80}")
    print(f"SPARK DISTRIBUTED TIME SERIES FORECASTING")
    print(f"{'='*80}")
    print(f"Metrics: {len(metrics)}")
    print(f"Groupings: {len(groupings)}")
    print(f"Cluster: 20 executors × 4 cores = 80 parallel workers")
    print(f"{'='*80}")

    start_time = time.time()

    # Step 1: Create tasks DataFrame (one row per metric/grouping/group combination)
    print("Step 1: Creating time series tasks...")
    tasks_df = create_time_series_tasks(spark_df, metrics, groupings)

    # Count tasks
    n_tasks = tasks_df.count()
    print(f"  Created {n_tasks} forecasting tasks")

    # Step 2: Repartition for distributed processing
    print(f"Step 2: Repartitioning for distributed processing...")
    n_partitions = min(n_tasks, 200)  # Max 200 partitions
    tasks_df = tasks_df.repartition(n_partitions)
    print(f"  Using {n_partitions} partitions")

    # Step 3: Apply distributed processing using RDD map
    print(f"Step 3: Distributing forecast function to executors...")
    print(f"Step 4: Running distributed forecasting...")
    print(f"  Processing {n_tasks} tasks across cluster...")

    # Use RDD map to distribute the work
    results_rdd = tasks_df.rdd.map(forecast_time_series_row)

    # Collect results
    print(f"Step 5: Collecting results...")
    results_list = results_rdd.collect()

    # Parse JSON results
    parsed_results = [json.loads(r) for r in results_list]

    end_time = time.time()
    duration = end_time - start_time

    print(f"{'='*80}")
    print(f"FORECASTING COMPLETE")
    print(f"{'='*80}")
    print(f"Total tasks: {len(parsed_results)}")
    print(f"Duration: {duration:.1f} seconds ({duration/60:.1f} minutes)")
    print(f"Throughput: {len(parsed_results) / duration:.1f} tasks/second")
    print(f"{'='*80}")

    return parsed_results

print("✓ Spark distributed orchestration function ready")

✓ Spark distributed orchestration function ready


In [52]:
# CONFIGURATION
# SparkCaster handles all distribution automatically

CONFIG = {
    'train_split': TRAIN_SPLIT,     # single source of truth (defined in the executor forecasting cell)
    'forecast_days': FORECAST_DAYS, # single source of truth (defined in the executor forecasting cell)
    'n_workers': 30,                # only used by the (legacy) PERFORMANCE ANALYSIS efficiency estimate
}

print("✓ Configuration set")
print(f"  Train/Test split: {CONFIG['train_split']}")
print(f"  Forecast horizon: {CONFIG['forecast_days']} days")

✓ Configuration set
  Train/Test split: 0.7
  Forecast horizon: 1100 days


In [53]:
# Note: Plot generation happens after results are collected
# No need for separate parallel plot generation with SparkCaster

print("✓ Plots will be generated from results after forecasting completes")

✓ Plots will be generated from results after forecasting completes


In [54]:
# SPARKCASTER MEMORY MANAGEMENT
# SparkCaster automatically handles memory across executors
# No need for manual chunking or memory optimization

print("✓ Memory managed automatically by Spark cluster")
print("  Each executor has 16GB RAM")
print("  No single-node memory bottlenecks")

✓ Memory managed automatically by Spark cluster
  Each executor has 16GB RAM
  No single-node memory bottlenecks


In [55]:
# SPARKCASTER IS NOW THE DEFAULT!
# This notebook uses SparkCaster for distributed processing
# No need for alternative implementations

print("✓ SparkCaster distributed processing is the default method")
print("  All forecasting tasks distributed across Spark cluster")

✓ SparkCaster distributed processing is the default method
  All forecasting tasks distributed across Spark cluster


In [56]:
# QUICK DIAGNOSTIC: Estimate workload before running
def estimate_workload():
    """
    Quick diagnostic to estimate processing time and resource needs
    """
    log_section("WORKLOAD ESTIMATION")
    
    # Estimate series count based on metrics and groupings
    # This is an approximation - actual count will be determined when tasks are created
    estimated_series = len(METRICS) * len(GROUPINGS)
    
    # For more detailed groupings, multiply by average groups per grouping
    # Rough estimates: product_resolved ~10, customer_segment ~5, region ~2, etc.
    avg_groups_per_grouping = 5
    estimated_series = estimated_series * avg_groups_per_grouping
    
    # Estimate processing time with SparkCaster
    # Assumptions: ~30 seconds per series, distributed across 80 workers
    est_time_per_series = 30  # seconds
    est_sequential_time = estimated_series * est_time_per_series
    
    # With 80 parallel workers on Spark cluster
    n_workers = 80  # 20 executors × 4 cores
    est_parallel_time = (est_sequential_time / n_workers) * 1.5  # 1.5x overhead for coordination
    
    log(f"\nDataset Analysis:")
    log(f"  Metrics: {len(METRICS)}")
    log(f"  Grouping strategies: {len(GROUPINGS)}")
    log(f"  Estimated time series: ~{estimated_series} (rough estimate)")
    log(f"  Models per series: 5")
    log(f"  Total model runs: ~{estimated_series * 5}")
    
    log(f"\nTime Estimates:")
    log(f"  Sequential processing: ~{est_sequential_time/60:.1f} minutes")
    log(f"  SparkCaster ({n_workers} workers): ~{est_parallel_time/60:.1f} minutes")
    log(f"  Speedup: ~{n_workers}x")
    
    log(f"\nCluster Resources:")
    log(f"  Executors: 20")
    log(f"  Cores per executor: 4")
    log(f"  Total parallel workers: {n_workers}")
    log(f"  Memory per executor: 16GB")
    
    log(f"\nRecommendation:")
    if estimated_series < 100:
        log(f"  ✓ Small dataset - SparkCaster will complete quickly")
        log(f"  ✓ Expected completion: {est_parallel_time/60:.1f} minutes")
    elif estimated_series < 500:
        log(f"  ✓ Medium dataset - perfect for SparkCaster")
        log(f"  ✓ Expected completion: {est_parallel_time/60:.1f} minutes")
    elif estimated_series < 2000:
        log(f"  ⚡ Large dataset - SparkCaster will handle this efficiently")
        log(f"  ⚡ Expected completion: {est_parallel_time/60:.1f} minutes")
    else:
        log(f"  ⚡⚡ Very large dataset - SparkCaster scales linearly")
        log(f"  ⚡⚡ Expected completion: {est_parallel_time/60:.1f} minutes")
        log(f"  💡 Tip: Monitor Spark UI for task progress")
    
    log("="*80)
    
    return estimated_series, est_parallel_time

# Run estimation
log("\nRunning workload estimation...\n")
estimated_series, estimated_time = estimate_workload()
log(f"\nProceed to next cell to start forecasting!")


Running workload estimation...

WORKLOAD ESTIMATION

Dataset Analysis:
  Metrics: 10
  Grouping strategies: 8
  Estimated time series: ~400 (rough estimate)
  Models per series: 5
  Total model runs: ~2000

Time Estimates:
  Sequential processing: ~200.0 minutes
  SparkCaster (80 workers): ~3.8 minutes
  Speedup: ~80x

Cluster Resources:
  Executors: 20
  Cores per executor: 4
  Total parallel workers: 80
  Memory per executor: 16GB

Recommendation:
  ✓ Medium dataset - perfect for SparkCaster
  ✓ Expected completion: 3.8 minutes

Proceed to next cell to start forecasting!


In [57]:
# RUN SPARKCASTER DISTRIBUTED FORECASTING
log_section("STARTING SPARKCASTER DISTRIBUTED FORECASTING")
log(f"Cluster: 20 executors × 4 cores = 80 parallel workers")
log("Estimated speedup: 50-100x faster than single-node processing!")
log("="*80)
log("")

# Run SparkCaster forecasting
import time
_forecast_start = time.time()
all_results = run_sparkcaster_forecasting(df, METRICS, GROUPINGS)
forecast_time = time.time() - _forecast_start  # wall-clock seconds; used by the PERFORMANCE ANALYSIS cell
all_results_df = pd.DataFrame(all_results)

log(f"✓ Forecasting complete!")
log(f"  Total results: {len(all_results_df)}")
log(f"  Successful: {len(all_results_df[all_results_df['status'] == 'completed'])}")
log(f"  Skipped: {len(all_results_df[all_results_df['status'] == 'skipped'])}")
log(f"  Errors: {len(all_results_df[all_results_df['status'] == 'error'])}")

# Build plot_data_list and all_plots for downstream cells
plot_data_list = []
all_plots = []

for result in all_results:
    if result.get('status') != 'completed':
        continue

    best_model = result.get('best_model')
    grouping = result.get('grouping') or result.get('grouping_name')

    metadata = {
        'metric': result.get('metric'),
        'grouping': grouping,
        'group_key': result.get('group_key'),
        'train_dates': pd.to_datetime(result.get('train_dates', [])),
        'test_dates': pd.to_datetime(result.get('test_dates', [])),
        'train_values': pd.Series(result.get('train_values', [])),
        'test_values': pd.Series(result.get('test_values', []))
    }

    results = {}
    for model_name, model_result in (result.get('results') or {}).items():
        if model_result.get('status') != 'success':
            continue

        results[model_name] = {
            'test_predictions': pd.Series(model_result.get('test_predictions', [])),
            'forecast': pd.Series(model_result.get('forecast', [])),
            'forecast_lower': pd.Series(model_result.get('forecast_lower', [])),
            'forecast_upper': pd.Series(model_result.get('forecast_upper', [])),
            'fitted': pd.Series(model_result.get('fitted', [])) if model_result.get('fitted') is not None else None,
            'metrics': model_result.get('metrics', {}),
            'metadata': metadata
        }

    if not results or best_model not in results:
        continue

    plot_entry = {
        'metric': result.get('metric'),
        'grouping': grouping,
        'group_key': result.get('group_key'),
        'best_model': best_model,
        'results': results,
    }

    plot_data_list.append(plot_entry)
    all_plots.append(plot_entry)



STARTING SPARKCASTER DISTRIBUTED FORECASTING
Cluster: 20 executors × 4 cores = 80 parallel workers
Estimated speedup: 50-100x faster than single-node processing!

SPARK DISTRIBUTED TIME SERIES FORECASTING
Metrics: 10
Groupings: 8
Cluster: 20 executors × 4 cores = 80 parallel workers
Step 1: Creating time series tasks...


  Created 620 forecasting tasks
Step 2: Repartitioning for distributed processing...
  Using 200 partitions
Step 3: Distributing forecast function to executors...
Step 4: Running distributed forecasting...
  Processing 620 tasks across cluster...
Step 5: Collecting results...
FORECASTING COMPLETE
Total tasks: 620
Duration: 30.4 seconds (0.5 minutes)
Throughput: 20.4 tasks/second
✓ Forecasting complete!
  Total results: 620
  Successful: 618
  Skipped: 0
  Errors: 2


In [58]:
# Check what errors occurred
# print("Sample errors:")
# for i, result in enumerate(all_results[:5]):
#     print(f"\n=== Task {i+1} ===")
#     print(f"Metric: {result.get('metric')}")
#     print(f"Grouping: {result.get('grouping_name')}")
#     print(f"Status: {result.get('status')}")
#     print(f"Error: {result.get('error', 'N/A')}")

In [59]:
# FORECAST QUALITY DIAGNOSTIC
# Run this after forecasting completes to check if warnings are a problem

from collections import Counter

log_section("FORECAST QUALITY ANALYSIS")

# Count statuses from results
statuses = []
model_success = Counter()

for result in all_results:
    if 'best_model' in result:
        model_success[result['best_model']] += 1

# Calculate metrics
total_series = len(plot_data_list)
successful_forecasts = len([p for p in plot_data_list if p is not None])
success_rate = (successful_forecasts / total_series * 100) if total_series > 0 else 0

log(f"\nOverall Statistics:")
log(f"  Total time series: {total_series}")
log(f"  Successful forecasts: {successful_forecasts}")
log(f"  Success rate: {success_rate:.1f}%")

log(f"\nBest Model Distribution:")
for model, count in model_success.most_common():
    pct = count / successful_forecasts * 100 if successful_forecasts > 0 else 0
    log(f"  {model}: {count} ({pct:.1f}%)")

log(f"\nQuality Assessment:")
if success_rate > 90:
    log("  ✅ EXCELLENT - Warnings are normal, no action needed")
    log("     Your error handling is working perfectly")
elif success_rate > 75:
    log("  🟢 GOOD - Most series forecasting successfully")
    log("     Consider adding warning suppression for cleaner output")
elif success_rate > 60:
    log("  🟡 FAIR - Some optimization recommended")
    log("     Review failed series and consider data quality checks")
else:
    log("  🔴 POOR - Investigation needed")
    log("     Significant data quality or model configuration issues")

log("\n" + "=" * 80)


FORECAST QUALITY ANALYSIS

Overall Statistics:
  Total time series: 618
  Successful forecasts: 618
  Success rate: 100.0%

Best Model Distribution:
  arima: 221 (35.8%)
  exponential_smoothing: 183 (29.6%)
  sarima: 117 (18.9%)
  holt_winters: 97 (15.7%)

Quality Assessment:
  ✅ EXCELLENT - Warnings are normal, no action needed
     Your error handling is working perfectly



In [60]:
# ── OUTPUT HELPERS (CoreWeave Object Storage / LOTA) ─────────────────────────
# Route 1: export the final pandas DataFrames straight to CoreWeave Object
# Storage as CSV + XLSX. No local files, no Iceberg. Download from Cloud Console.
import io, os
import pandas as pd
import fsspec   # s3fs / fsspec / openpyxl come from the driver bootstrap cell

# --- Config ---
S3_BUCKET   = 'jbok-sandbox-test'
S3_ENDPOINT = 'http://cwlota.com'          # LOTA base; virtual-hosted -> http://<bucket>.cwlota.com
S3_REGION   = 'US-EAST-04A'

# One run-scoped UTC timestamp so all outputs share a single folder
from datetime import datetime, timezone
RUN_TS    = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
S3_PREFIX = f'jbok/time-series-fcst-sparkcaster/{RUN_TS}/'

# Access key / secret you created. Uses kernel env vars if set, otherwise prompts
# (getpass keeps the secret out of the notebook file). NOTE: a .env in your
# workspace is NOT readable here — this code runs in the remote Spark kernel,
# which does not mount /home/coreweave/jbok-cw.
S3_ACCESS_KEY = os.environ.get('CW_S3_ACCESS_KEY')
S3_SECRET_KEY = os.environ.get('CW_S3_SECRET_KEY')
if not S3_ACCESS_KEY or not S3_SECRET_KEY:
    import getpass
    S3_ACCESS_KEY = S3_ACCESS_KEY or getpass.getpass('CoreWeave S3 access key: ')
    S3_SECRET_KEY = S3_SECRET_KEY or getpass.getpass('CoreWeave S3 secret key: ')

STORAGE_OPTIONS = {
    'key': S3_ACCESS_KEY,
    'secret': S3_SECRET_KEY,
    'client_kwargs': {'endpoint_url': S3_ENDPOINT, 'region_name': S3_REGION},
    'config_kwargs': {'s3': {'addressing_style': 'virtual'}},   # LOTA is virtual-hosted
}

_S3_SAVED = []   # (label, s3_uri) collected across the run

def _s3_uri(name, ext):
    return f"s3://{S3_BUCKET}/{S3_PREFIX}{name}.{ext}"

def _put_bytes(uri, data):
    with fsspec.open(uri, 'wb', **STORAGE_OPTIONS) as f:
        f.write(data)

def save_df_to_s3(name, pdf, formats=('csv', 'xlsx')):
    """Write a pandas DataFrame to Object Storage as CSV and/or XLSX."""
    if pdf is None or len(pdf) == 0:
        print(f"  (skipping '{name}': empty DataFrame)")
        return
    formats = list(formats)
    if 'xlsx' in formats and len(pdf) > 1_048_575:      # Excel row limit
        print(f"  (xlsx skipped for '{name}': {len(pdf):,} rows exceed Excel's limit; CSV only)")
        formats = [f for f in formats if f != 'xlsx']
    if 'csv' in formats:
        uri = _s3_uri(name, 'csv')
        _put_bytes(uri, pdf.to_csv(index=False).encode('utf-8'))
        _S3_SAVED.append((f'{name} (csv)', uri)); print(f"  → {uri}")
    if 'xlsx' in formats:
        uri = _s3_uri(name, 'xlsx')
        buf = io.BytesIO()
        with pd.ExcelWriter(buf, engine='openpyxl') as xw:
            pdf.to_excel(xw, index=False, sheet_name=(name[:31] or 'Sheet1'))
        _put_bytes(uri, buf.getvalue())
        _S3_SAVED.append((f'{name} (xlsx)', uri)); print(f"  → {uri}")

def print_s3_manifest():
    """Print every object written this run, with its s3:// path."""
    print(f"Bucket  : {S3_BUCKET}")
    print(f"Prefix  : {S3_PREFIX}")
    print(f"Endpoint: {S3_ENDPOINT} (virtual-hosted)")
    if not _S3_SAVED:
        print("(no objects written this run)")
        return
    for label, uri in _S3_SAVED:
        print(f"  {label:52} {uri}")

print(f"✓ Object-storage export helpers ready. Target: s3://{S3_BUCKET}/{S3_PREFIX}")


✓ Object-storage export helpers ready. Target: s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/20260723_221127/


In [61]:
# Create and save results to Excel files

# 1. Create DataFrame for ALL models (one row per model per time series)
if len(all_results) == 0:
    print("\n⚠️  WARNING: No forecast results available!")
    print("This likely means the forecasting cell didn't run or encountered errors.")
    print("Please run the forecasting cell first (the cell that calls run_sparkcaster_forecasting).")
    df_all_models = pd.DataFrame()  # Empty DataFrame
else:
    rows = []
    for result in all_results:
        if result.get('status') != 'completed':
            continue

        metric = result.get('metric')
        grouping = result.get('grouping') or result.get('grouping_name')
        group_key = result.get('group_key')
        best_model = result.get('best_model')

        for model_name, model_result in (result.get('results') or {}).items():
            if model_result.get('status') != 'success':
                continue

            metrics = model_result.get('metrics', {})
            rows.append({
                'metric': metric,
                'grouping': grouping,
                'group_key': group_key,
                'model': model_name,
                'is_best': model_name == best_model,
                'MSE': metrics.get('MSE'),
                'RMSE': metrics.get('RMSE'),
                'MAPE': metrics.get('MAPE'),
                'train_size': result.get('train_size'),
                'test_size': result.get('test_size'),
                'status': model_result.get('status')
            })

    df_all_models = pd.DataFrame(rows)

    print("\nAll Models Results:")
    print(f"Shape: {df_all_models.shape}")
    print(f"Columns: {list(df_all_models.columns)}")
    print("Sample data:")
    print(df_all_models.head(10))

# Save to Excel — via sandbox parquet round-trip, then read back and save locally
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Export to CoreWeave Object Storage (CSV + XLSX)
save_df_to_s3('all_models_results', df_all_models)



All Models Results:
Shape: (2225, 11)
Columns: ['metric', 'grouping', 'group_key', 'model', 'is_best', 'MSE', 'RMSE', 'MAPE', 'train_size', 'test_size', 'status']
Sample data:
            metric                         grouping group_key  \
0     gpu_util_p50                 customer_segment    AI Lab   
1     gpu_util_p50                 customer_segment    AI Lab   
2     gpu_util_p50                 customer_segment    AI Lab   
3     gpu_util_p50                 customer_segment    AI Lab   
4     gpu_util_p50                 product_resolved    LEGACY   
5     gpu_util_p50                 product_resolved    LEGACY   
6     gpu_util_p50                 product_resolved    LEGACY   
7  tensor_util_p50  region_summary+product_resolved  NAM_H100   
8  tensor_util_p50  region_summary+product_resolved  NAM_H100   
9  tensor_util_p50  region_summary+product_resolved  NAM_H100   

                   model  is_best       MSE      RMSE       MAPE  train_size  \
0  exponential_smoothing   

In [62]:
# 2. Create DataFrame for BEST models only
if df_all_models.empty:
    print("\n⚠️  WARNING: df_all_models is empty, cannot create best models DataFrame.")
    print("Please run the forecasting cell first.")
    df_best_models = pd.DataFrame()  # Empty DataFrame
elif 'is_best' not in df_all_models.columns:
    print("\n⚠️  WARNING: 'is_best' column not found in results.")
    print("This suggests the forecasting completed but didn't include best model selection.")
    df_best_models = pd.DataFrame()  # Empty DataFrame
else:
    df_best_models = df_all_models[df_all_models['is_best'] == True].copy()
    
    print("\nBest Models Results:")
    print(f"Shape: {df_best_models.shape}")
    print(f"\nSample data:")
    print(df_best_models.head(10))
    
    # Save to Excel
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# Export to CoreWeave Object Storage (CSV + XLSX)
save_df_to_s3('best_models_results', df_best_models)


Best Models Results:
Shape: (618, 11)

Sample data:
                     metric                          grouping  group_key  \
2              gpu_util_p50                  customer_segment     AI Lab   
4              gpu_util_p50                  product_resolved     LEGACY   
9           tensor_util_p50   region_summary+product_resolved   NAM_H100   
13     chip_power_fleet_p95                  product_resolved       H200   
18             gpu_util_p95                   product_segment        HGX   
21  tensor_tflops_fleet_p95   region_summary+product_resolved   NAM_B300   
23             gpu_util_p95                  product_resolved      GB300   
26  tensor_tflops_fleet_p95  product_segment+product_resolved  MGX_GB200   
30     chip_power_fleet_p95   region_summary+product_resolved   NAM_H200   
33             gpu_util_p95                  product_resolved       B300   

                    model  is_best           MSE           RMSE       MAPE  \
2                  sarima     Tr

In [63]:
from pathlib import Path
import os

print("cwd:", os.getcwd())
print("home:", str(Path.home()))

for p in [os.getcwd(), str(Path.home()), "/home/coreweave", "/tmp"]:
    print(p, "exists=", os.path.isdir(p), "writable=", os.access(p, os.W_OK))


cwd: /opt/spark/work-dir
home: /home/spark
/opt/spark/work-dir exists= True writable= True
/home/spark exists= True writable= True
/home/coreweave exists= False writable= False
/tmp exists= True writable= True


In [64]:
import os, glob
print("cwd:", os.getcwd())
print(glob.glob(os.path.join(os.getcwd(), "time_series_all_models_results_*.xlsx")))
print(glob.glob(os.path.join(str(Path.home()), "time_series_all_models_results_*.xlsx")))


cwd: /opt/spark/work-dir
[]
[]


In [65]:
# 3. Extract and save FULL DAILY FORECAST VALUES for best models
# This creates a detailed CSV with one row per day per time series

log_section("EXTRACTING FULL DAILY FORECAST VALUES")

forecast_details = []

for plot in plot_data_list:
    if plot is None:
        continue

    best_model = plot['best_model']
    best_result = plot['results'][best_model]

    # Get forecast array (point estimate)
    forecast_array = best_result.get('forecast')
    forecast_array = forecast_array.values if hasattr(forecast_array, 'values') else forecast_array

    # Get prediction intervals (lower and upper bounds)
    forecast_lower = best_result.get('forecast_lower')
    forecast_upper = best_result.get('forecast_upper')
    forecast_lower = forecast_lower.values if hasattr(forecast_lower, 'values') else forecast_lower
    forecast_upper = forecast_upper.values if hasattr(forecast_upper, 'values') else forecast_upper

    # Coerce to numpy arrays and align lengths
    forecast_array = np.asarray(forecast_array) if forecast_array is not None else np.array([])
    forecast_lower = np.asarray(forecast_lower) if forecast_lower is not None else np.array([])
    forecast_upper = np.asarray(forecast_upper) if forecast_upper is not None else np.array([])

    # If intervals are missing or length-mismatched, fall back to NaN arrays
    n = len(forecast_array)
    if len(forecast_lower) != n:
        forecast_lower = np.full(n, np.nan)
    if len(forecast_upper) != n:
        forecast_upper = np.full(n, np.nan)

    # Apply floor to prevent negative values (ignore NaNs)
    forecast_array = np.maximum(0, forecast_array)
    forecast_lower = np.where(np.isnan(forecast_lower), forecast_lower, np.maximum(0, forecast_lower))
    forecast_upper = np.where(np.isnan(forecast_upper), forecast_upper, np.maximum(0, forecast_upper))

    # Get metadata
    metadata = best_result['metadata']
    last_historical_date = metadata['train_dates'].max()

    # Create date range for forecast
    forecast_dates = pd.date_range(
        start=last_historical_date + pd.Timedelta(days=1),
        periods=n,
        freq='D'
    )

    # Create DataFrame for this time series forecast with prediction intervals
    df_forecast = pd.DataFrame({
        'metric': plot['metric'],
        'grouping': plot['grouping'],
        'group_key': plot['group_key'],
        'model': best_model,
        'forecast_date': forecast_dates,
        'forecast_value': forecast_array,  # Keep for backward compatibility
        'forecast_p50': forecast_array,    # Point estimate (median)
        'forecast_p10': forecast_lower,    # Lower confidence bound (10th percentile)
        'forecast_p90': forecast_upper,    # Upper confidence bound (90th percentile)
        'last_historical_date': last_historical_date,
        'forecast_horizon_days': range(1, n + 1)
    })

    forecast_details.append(df_forecast)

# Combine all forecasts
if forecast_details:
    df_all_forecasts = pd.concat(forecast_details, ignore_index=True)
else:
    df_all_forecasts = pd.DataFrame()

log("\nForecast Details:")
log(f"  Total rows: {len(df_all_forecasts):,}")
if not df_all_forecasts.empty:
    log(f"  Unique metrics: {df_all_forecasts['metric'].nunique()}")
    log(f"  Date range: {df_all_forecasts['forecast_date'].min()} to {df_all_forecasts['forecast_date'].max()}")

    log("\nSample data (with prediction intervals):")
    log(df_all_forecasts.head(10))

    # Save to CSV
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    # Export to CoreWeave Object Storage (CSV + XLSX)
    save_df_to_s3('forecast_daily_values', df_all_forecasts)
    log(f"\n✓ Saved {len(df_all_forecasts):,} daily forecast values")
    log(f"  Columns: {', '.join(df_all_forecasts.columns.tolist())}")
else:
    log("  No forecast details were generated.")

# Show utilization metrics specifically
log_section("UTILIZATION METRICS FORECAST RANGES")

if not df_all_forecasts.empty:
    util_forecasts = df_all_forecasts[df_all_forecasts['metric'].str.contains('util', case=False, na=False)]
    if len(util_forecasts) > 0:
        util_summary = util_forecasts.groupby(['metric', 'grouping', 'group_key']).agg({
            'forecast_p50': ['min', 'max', 'mean'],
            'forecast_p10': ['min', 'max', 'mean'],
            'forecast_p90': ['min', 'max', 'mean']
        }).round(4)

        log(f"Utilization forecast summary (P10/P50/P90):")
        log(util_summary.head(20))

        # Check bounds for all percentiles (ignore NaNs)
        max_p90 = util_forecasts['forecast_p90'].max()
        min_p10 = util_forecasts['forecast_p10'].min()
        max_p50 = util_forecasts['forecast_p50'].max()

        log_section("BOUNDS CHECK (with prediction intervals)")
        log(f"Global min forecast P10 value: {min_p10:.6f}")
        log(f"Global max forecast P50 value: {max_p50:.6f}")
        log(f"Global max forecast P90 value: {max_p90:.6f}")

        if max_p90 > 1.0:
            exceeds = util_forecasts[util_forecasts['forecast_p90'] > 1.0]
            log(f"⚠️  WARNING: {len(exceeds):,} P90 values exceed 1.0 (need to restart kernel and re-run)")
        else:
            log(f"✓ All utilization P90 forecasts within [0, 1] bounds")

        if min_p10 < 0.0:
            below = util_forecasts[util_forecasts['forecast_p10'] < 0.0]
            log(f"⚠️  WARNING: {len(below):,} P10 values below 0.0 (need to restart kernel and re-run)")
        else:
            log(f"✓ All utilization P10 forecasts >= 0.0")
    else:
        log("No utilization metrics found in forecast data")
else:
    log("No utilization metrics found in forecast data")

log("")
log_section(f"✓ Daily forecast extraction complete (with P10/P50/P90 prediction intervals)")


EXTRACTING FULL DAILY FORECAST VALUES



Forecast Details:
  Total rows: 679,800
  Unique metrics: 10
  Date range: 2026-01-19 00:00:00 to 2029-06-04 00:00:00

Sample data (with prediction intervals):
         metric          grouping group_key   model forecast_date  \
0  gpu_util_p50  customer_segment    AI Lab  sarima    2026-01-19   
1  gpu_util_p50  customer_segment    AI Lab  sarima    2026-01-20   
2  gpu_util_p50  customer_segment    AI Lab  sarima    2026-01-21   
3  gpu_util_p50  customer_segment    AI Lab  sarima    2026-01-22   
4  gpu_util_p50  customer_segment    AI Lab  sarima    2026-01-23   
5  gpu_util_p50  customer_segment    AI Lab  sarima    2026-01-24   
6  gpu_util_p50  customer_segment    AI Lab  sarima    2026-01-25   
7  gpu_util_p50  customer_segment    AI Lab  sarima    2026-01-26   
8  gpu_util_p50  customer_segment    AI Lab  sarima    2026-01-27   
9  gpu_util_p50  customer_segment    AI Lab  sarima    2026-01-28   

   forecast_value  forecast_p50  forecast_p10  forecast_p90  \
0        0.39829

In [66]:
# 3b. Extract DAILY FORECAST VALUES for ALL MODELS
# This creates a detailed DataFrame with one row per day per time series per model

log_section("EXTRACTING DAILY FORECAST VALUES FOR ALL MODELS")

forecast_details_all = []

for plot in plot_data_list:
    if plot is None:
        continue

    # Loop through ALL models, not just the best one
    for model_name, model_result in plot['results'].items():

        # Get forecast array (point estimate)
        forecast_array = model_result.get('forecast')
        forecast_array = forecast_array.values if hasattr(forecast_array, 'values') else forecast_array

        # Get prediction intervals (lower and upper bounds)
        forecast_lower = model_result.get('forecast_lower')
        forecast_upper = model_result.get('forecast_upper')
        forecast_lower = forecast_lower.values if hasattr(forecast_lower, 'values') else forecast_lower
        forecast_upper = forecast_upper.values if hasattr(forecast_upper, 'values') else forecast_upper

        # Coerce to numpy arrays and align lengths
        forecast_array = np.asarray(forecast_array) if forecast_array is not None else np.array([])
        forecast_lower = np.asarray(forecast_lower) if forecast_lower is not None else np.array([])
        forecast_upper = np.asarray(forecast_upper) if forecast_upper is not None else np.array([])

        n = len(forecast_array)
        if len(forecast_lower) != n:
            forecast_lower = np.full(n, np.nan)
        if len(forecast_upper) != n:
            forecast_upper = np.full(n, np.nan)

        # Apply floor to prevent negative values (ignore NaNs)
        forecast_array = np.maximum(0, forecast_array)
        forecast_lower = np.where(np.isnan(forecast_lower), forecast_lower, np.maximum(0, forecast_lower))
        forecast_upper = np.where(np.isnan(forecast_upper), forecast_upper, np.maximum(0, forecast_upper))

        # Get metadata
        metadata = model_result['metadata']
        last_historical_date = metadata['train_dates'].max()

        # Create date range for forecast
        forecast_dates = pd.date_range(
            start=last_historical_date + pd.Timedelta(days=1),
            periods=n,
            freq='D'
        )

        # Create DataFrame for this time series forecast with prediction intervals
        df_forecast = pd.DataFrame({
            'metric': plot['metric'],
            'grouping': plot['grouping'],
            'group_key': plot['group_key'],
            'model': model_name,
            'is_best_model': (model_name == plot['best_model']),
            'forecast_date': forecast_dates,
            'forecast_value': forecast_array,  # Keep for backward compatibility
            'forecast_p50': forecast_array,    # Point estimate (median)
            'forecast_p10': forecast_lower,    # Lower confidence bound (10th percentile)
            'forecast_p90': forecast_upper,    # Upper confidence bound (90th percentile)
            'last_historical_date': last_historical_date,
            'forecast_horizon_days': range(1, n + 1)
        })

        forecast_details_all.append(df_forecast)

# Combine all forecasts from all models
if forecast_details_all:
    df_all_models_forecasts = pd.concat(forecast_details_all, ignore_index=True)
else:
    df_all_models_forecasts = pd.DataFrame()

log(f"All Models Forecast Details:")
log(f"  Total rows: {len(df_all_models_forecasts):,}")
if not df_all_models_forecasts.empty:
    log(f"  Unique time series: {len(df_all_models_forecasts.groupby(['metric', 'grouping', 'group_key']))}")
    log(f"  Unique models: {df_all_models_forecasts['model'].nunique()}")
    log(f"  Models: {sorted(df_all_models_forecasts['model'].unique())}")
    log(f"  Unique metrics: {df_all_models_forecasts['metric'].nunique()}")
    log(f"  Date range: {df_all_models_forecasts['forecast_date'].min()} to {df_all_models_forecasts['forecast_date'].max()}")

    log(f"Model breakdown:")
    log(df_all_models_forecasts.groupby('model').size())

    log(f"Sample data (with all models):")
    log(df_all_models_forecasts.head(15))

log_section(f"✓ All models daily forecast extraction complete")


EXTRACTING DAILY FORECAST VALUES FOR ALL MODELS
All Models Forecast Details:
  Total rows: 2,447,500
  Unique time series: 618
  Unique models: 4
  Models: ['arima', 'exponential_smoothing', 'holt_winters', 'sarima']
  Unique metrics: 10
  Date range: 2026-01-19 00:00:00 to 2029-06-04 00:00:00
Model breakdown:
model
arima                    679800
exponential_smoothing    679800
holt_winters             408100
sarima                   679800
dtype: int64
Sample data (with all models):
          metric          grouping group_key                  model  \
0   gpu_util_p50  customer_segment    AI Lab  exponential_smoothing   
1   gpu_util_p50  customer_segment    AI Lab  exponential_smoothing   
2   gpu_util_p50  customer_segment    AI Lab  exponential_smoothing   
3   gpu_util_p50  customer_segment    AI Lab  exponential_smoothing   
4   gpu_util_p50  customer_segment    AI Lab  exponential_smoothing   
5   gpu_util_p50  customer_segment    AI Lab  exponential_smoothing   
6   gpu_util_

In [67]:
# 3c. AGGREGATE DAILY FORECASTS TO MONTHLY GRAIN - ALL MODELS
# This creates a CSV with monthly aggregated forecast values for ALL models

log("")
log_section("AGGREGATING DAILY FORECASTS TO MONTHLY GRAIN - ALL MODELS")

# Add year-month column for grouping
df_all_models_forecasts['year_month'] = df_all_models_forecasts['forecast_date'].dt.to_period('M')

# Group by time series identifiers, MODEL, and year-month, then aggregate
monthly_forecasts_all = df_all_models_forecasts.groupby([
    'metric', 
    'grouping', 
    'group_key', 
    'model',
    'year_month'
]).agg({
    'is_best_model': 'first',  # Boolean flag - same for all days in month
    'forecast_value': 'mean',  # Average daily values for the month (backward compatibility)
    'forecast_p50': 'mean',    # Average P50 point estimates for the month
    'forecast_p10': 'mean',    # Average P10 lower bounds for the month
    'forecast_p90': 'mean',    # Average P90 upper bounds for the month
    'forecast_date': ['min', 'max'],  # First and last date in month
    'last_historical_date': 'first',
    'forecast_horizon_days': ['min', 'max']  # Min and max horizon days in month
}).reset_index()

# Flatten column names
monthly_forecasts_all.columns = [
    'metric', 'grouping', 'group_key', 'model', 'year_month',
    'is_best_model',
    'avg_forecast_value',
    'avg_forecast_p50', 
    'avg_forecast_p10', 
    'avg_forecast_p90',
    'month_start_date', 'month_end_date',
    'last_historical_date',
    'forecast_horizon_days_min', 'forecast_horizon_days_max'
]

# Convert year_month back to string for better CSV readability
monthly_forecasts_all['year_month'] = monthly_forecasts_all['year_month'].astype(str)

# Add a column for number of days in the forecast month period
monthly_forecasts_all['days_in_month_period'] = (
    pd.to_datetime(monthly_forecasts_all['month_end_date']) - 
    pd.to_datetime(monthly_forecasts_all['month_start_date'])
).dt.days + 1

log(f"\nMonthly Forecast Summary (All Models):")
log(f"  Total rows: {len(monthly_forecasts_all):,}")
log(f"  Unique time series: {len(monthly_forecasts_all.groupby(['metric', 'grouping', 'group_key']))}")
log(f"  Unique models: {monthly_forecasts_all['model'].nunique()}")
log(f"  Models: {sorted(monthly_forecasts_all['model'].unique())}")
log(f"  Unique metrics: {monthly_forecasts_all['metric'].nunique()}")
log(f"  Date range: {monthly_forecasts_all['year_month'].min()} to {monthly_forecasts_all['year_month'].max()}")

log(f"\nRows per model:")
log(monthly_forecasts_all.groupby('model').size())

log(f"\nBest model flags:")
log(monthly_forecasts_all.groupby('is_best_model').size())

log(f"\nSample monthly data (all models):")
log(monthly_forecasts_all.head(20))

# Save to CSV
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# Export to CoreWeave Object Storage (CSV + XLSX)
save_df_to_s3('forecast_monthly_values_all', monthly_forecasts_all)
log(f"\n✓ Saved {len(monthly_forecasts_all):,} monthly forecast values (all models)")

# Show comparison between best model only vs all models (if available)
try:
    log_section("COMPARISON: BEST MODEL ONLY vs ALL MODELS")
    log(f"Best model only CSV rows: {len(monthly_forecasts):,}")
    log(f"All models CSV rows: {len(monthly_forecasts_all):,}")
    log(f"Ratio: {len(monthly_forecasts_all) / len(monthly_forecasts):.2f}x more rows")
except NameError:
    log_section("ALL MODELS CSV CREATED")
    log(f"All models CSV rows: {len(monthly_forecasts_all):,}")
    log("(Run cell 36 to create best-model-only CSV for comparison)")

# Show utilization metrics at monthly grain for all models
log("")
log_section("UTILIZATION METRICS - MONTHLY AGGREGATION (ALL MODELS)")

util_monthly_all = monthly_forecasts_all[monthly_forecasts_all['metric'].str.contains('util', case=False)]
if len(util_monthly_all) > 0:
    log(f"\nMonthly utilization forecasts (all models):")
    log(f"  Rows: {len(util_monthly_all):,}")
    log(f"  Models: {sorted(util_monthly_all['model'].unique())}")
    
    # Check bounds
    max_p50 = util_monthly_all['avg_forecast_p50'].max()
    min_p50 = util_monthly_all['avg_forecast_p50'].min()
    max_p10 = util_monthly_all['avg_forecast_p10'].max()
    min_p10 = util_monthly_all['avg_forecast_p10'].min()
    max_p90 = util_monthly_all['avg_forecast_p90'].max()
    min_p90 = util_monthly_all['avg_forecast_p90'].min()
    
    log("")
    log_section("MONTHLY BOUNDS CHECK (ALL MODELS)")
    log(f"P50 (point estimate) range: [{min_p50:.6f}, {max_p50:.6f}]")
    log(f"P10 (lower bound) range:    [{min_p10:.6f}, {max_p10:.6f}]")
    log(f"P90 (upper bound) range:    [{min_p90:.6f}, {max_p90:.6f}]")
    
    # Check P50/P10/P90 bounds
    if max_p50 > 1.0:
        exceeds = util_monthly_all[util_monthly_all['avg_forecast_p50'] > 1.0]
        log(f"⚠️  WARNING: {len(exceeds):,} monthly P50 values exceed 1.0")
        log(f"   Models with issues: {sorted(exceeds['model'].unique())}")
    else:
        log(f"✓ All monthly P50 utilization forecasts within [0, 1] bounds")
    
    if min_p50 < 0.0:
        below = util_monthly_all[util_monthly_all['avg_forecast_p50'] < 0.0]
        log(f"⚠️  WARNING: {len(below):,} monthly P50 values below 0.0")
        log(f"   Models with issues: {sorted(below['model'].unique())}")
    else:
        log(f"✓ All monthly P50 utilization forecasts >= 0.0")
    
    if max_p90 > 1.0:
        exceeds = util_monthly_all[util_monthly_all['avg_forecast_p90'] > 1.0]
        log(f"⚠️  WARNING: {len(exceeds):,} monthly P90 values exceed 1.0")
        log(f"   Models with issues: {sorted(exceeds['model'].unique())}")
    else:
        log(f"✓ All monthly P90 utilization forecasts within [0, 1] bounds")
    
    if min_p10 < 0.0:
        below = util_monthly_all[util_monthly_all['avg_forecast_p10'] < 0.0]
        log(f"⚠️  WARNING: {len(below):,} monthly P10 values below 0.0")
        log(f"   Models with issues: {sorted(below['model'].unique())}")
    else:
        log(f"✓ All monthly P10 utilization forecasts >= 0.0")
else:
    log("No utilization metrics found in monthly forecast data")

log(f"\n{'='*80}")
log(f"✓ Monthly forecast aggregation complete (ALL MODELS)")
log(f"  Output: s3://{S3_BUCKET}/{S3_PREFIX}forecast_monthly_values_all.csv")
log(f"{'='*80}")


AGGREGATING DAILY FORECASTS TO MONTHLY GRAIN - ALL MODELS



Monthly Forecast Summary (All Models):
  Total rows: 82,442
  Unique time series: 618
  Unique models: 4
  Models: ['arima', 'exponential_smoothing', 'holt_winters', 'sarima']
  Unique metrics: 10
  Date range: 2026-01 to 2029-06

Rows per model:
model
arima                    22896
exponential_smoothing    22896
holt_winters             13754
sarima                   22896
dtype: int64

Best model flags:
is_best_model
False    59546
True     22896
dtype: int64

Sample monthly data (all models):
                  metric grouping group_key  model year_month  is_best_model  \
0   chip_power_fleet_p50      All       All  arima    2026-01          False   
1   chip_power_fleet_p50      All       All  arima    2026-02          False   
2   chip_power_fleet_p50      All       All  arima    2026-03          False   
3   chip_power_fleet_p50      All       All  arima    2026-04          False   
4   chip_power_fleet_p50      All       All  arima    2026-05          False   
5   chip_power_fle

In [68]:
# 3d. ADD HISTORICAL DATA TO MONTHLY CSV (ALL MODELS)
# This extends the monthly CSV to include actual historical values before forecasts

log("")
log_section("ADDING HISTORICAL DATA TO MONTHLY FORECASTS - ALL MODELS")

historical_monthly = []

# Extract historical data from plot_data_list
for plot in plot_data_list:
    if plot is None:
        continue
    
    # Get historical data from any model's metadata (same for all models)
    first_model = list(plot['results'].keys())[0]
    metadata = plot['results'][first_model]['metadata']
    
    train_dates = pd.to_datetime(metadata['train_dates'])
    test_dates = pd.to_datetime(metadata.get('test_dates', []))
    train_values = pd.Series(metadata['train_values'])
    test_values = pd.Series(metadata.get('test_values', []))

    # Combine train + test to get full historical actuals
    full_dates = pd.concat([pd.Series(train_dates), pd.Series(test_dates)], ignore_index=True)
    full_values = pd.concat([train_values, test_values], ignore_index=True)

    # Create DataFrame with historical data
    df_hist = pd.DataFrame({
        'date': full_dates,
        'value': full_values,
        'metric': plot['metric'],
        'grouping': plot['grouping'],
        'group_key': plot['group_key']
    })
    
    # Add year-month column
    df_hist['year_month'] = pd.to_datetime(df_hist['date']).dt.to_period('M')
    
    # Group by month and calculate monthly averages
    monthly_hist = df_hist.groupby([
        'metric', 'grouping', 'group_key', 'year_month'
    ]).agg({
        'value': 'mean',
        'date': ['min', 'max']
    }).reset_index()
    
    # Flatten columns
    monthly_hist.columns = [
        'metric', 'grouping', 'group_key', 'year_month',
        'avg_actual_value', 'month_start_date', 'month_end_date'
    ]
    
    historical_monthly.append(monthly_hist)

# Combine all historical data
df_historical_monthly = pd.concat(historical_monthly, ignore_index=True)
df_historical_monthly['year_month'] = df_historical_monthly['year_month'].astype(str)

log(f"\nHistorical Monthly Data:")
log(f"  Total rows: {len(df_historical_monthly):,}")
log(f"  Unique time series: {len(df_historical_monthly.groupby(['metric', 'grouping', 'group_key']))}")
log(f"  Date range: {df_historical_monthly['year_month'].min()} to {df_historical_monthly['year_month'].max()}")

# Now merge with forecast data
# For historical data, we need to add model column and replicate for each model
models_list = monthly_forecasts_all['model'].unique()
log(f"\nReplicating historical data for {len(models_list)} models: {sorted(models_list)}")

# Replicate historical data for each model
historical_replicated = []
for model in models_list:
    df_hist_model = df_historical_monthly.copy()
    df_hist_model['model'] = model
    historical_replicated.append(df_hist_model)

df_historical_all_models = pd.concat(historical_replicated, ignore_index=True)

# Add columns to match forecast structure
df_historical_all_models['is_best_model'] = False  # Not applicable for historical
df_historical_all_models['avg_forecast_value'] = None
df_historical_all_models['avg_forecast_p50'] = None
df_historical_all_models['avg_forecast_p10'] = None
df_historical_all_models['avg_forecast_p90'] = None
df_historical_all_models['last_historical_date'] = None
df_historical_all_models['forecast_horizon_days_min'] = None
df_historical_all_models['forecast_horizon_days_max'] = None
df_historical_all_models['days_in_month_period'] = (
    pd.to_datetime(df_historical_all_models['month_end_date']) - 
    pd.to_datetime(df_historical_all_models['month_start_date'])
).dt.days + 1
df_historical_all_models['data_type'] = 'actual'

# Add data_type to forecast data
monthly_forecasts_all['avg_actual_value'] = None
monthly_forecasts_all['data_type'] = 'forecast'

# Ensure column order matches
common_columns = [
    'metric', 'grouping', 'group_key', 'model', 'year_month',
    'is_best_model', 'data_type',
    'avg_actual_value', 'avg_forecast_value',
    'avg_forecast_p50', 'avg_forecast_p10', 'avg_forecast_p90',
    'month_start_date', 'month_end_date',
    'last_historical_date',
    'forecast_horizon_days_min', 'forecast_horizon_days_max',
    'days_in_month_period'
]

# Reorder columns
df_historical_all_models = df_historical_all_models[common_columns]
monthly_forecasts_all_ordered = monthly_forecasts_all[common_columns]

# Combine historical + forecast
monthly_forecasts_with_history = pd.concat([
    df_historical_all_models,
    monthly_forecasts_all_ordered
], ignore_index=True)

# Sort by time series, model, and date
monthly_forecasts_with_history = monthly_forecasts_with_history.sort_values([
    'metric', 'grouping', 'group_key', 'model', 'year_month'
]).reset_index(drop=True)

log(f"\nCombined Historical + Forecast Data:")
log(f"  Total rows: {len(monthly_forecasts_with_history):,}")
log(f"  Historical rows: {len(monthly_forecasts_with_history[monthly_forecasts_with_history['data_type']=='actual']):,}")
log(f"  Forecast rows: {len(monthly_forecasts_with_history[monthly_forecasts_with_history['data_type']=='forecast']):,}")
log(f"  Date range: {monthly_forecasts_with_history['year_month'].min()} to {monthly_forecasts_with_history['year_month'].max()}")

log(f"\nSample data (showing transition from actual to forecast):")
# Show sample for one time series
sample_ts = monthly_forecasts_with_history.head(50)
log(sample_ts[['metric', 'grouping', 'group_key', 'model', 'year_month', 'data_type', 'avg_actual_value', 'avg_forecast_p50']].head(20))

# Save combined file
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# Export to CoreWeave Object Storage (CSV + XLSX)
save_df_to_s3('forecast_monthly_values_all_with_history', monthly_forecasts_with_history)
log(f"\n✓ Saved {len(monthly_forecasts_with_history):,} monthly values (historical + forecast, all models)")

log(f"\n{'='*80}")
log(f"✓ Historical + Forecast monthly data creation complete")
log(f"  - Use 'data_type' column to filter: 'actual' vs 'forecast'")
log(f"  - Historical data: avg_actual_value column")
log(f"  - Forecast data: avg_forecast_p50, avg_forecast_p10, avg_forecast_p90 columns")
log(f"{'='*80}")



ADDING HISTORICAL DATA TO MONTHLY FORECASTS - ALL MODELS

Historical Monthly Data:
  Total rows: 9,445
  Unique time series: 618
  Date range: 2025-01 to 2026-06

Replicating historical data for 4 models: ['arima', 'exponential_smoothing', 'holt_winters', 'sarima']

Combined Historical + Forecast Data:
  Total rows: 120,222
  Historical rows: 37,780
  Forecast rows: 82,442
  Date range: 2025-01 to 2029-06

Sample data (showing transition from actual to forecast):
                  metric grouping group_key  model year_month data_type  \
0   chip_power_fleet_p50      All       All  arima    2025-01    actual   
1   chip_power_fleet_p50      All       All  arima    2025-02    actual   
2   chip_power_fleet_p50      All       All  arima    2025-03    actual   
3   chip_power_fleet_p50      All       All  arima    2025-04    actual   
4   chip_power_fleet_p50      All       All  arima    2025-05    actual   
5   chip_power_fleet_p50      All       All  arima    2025-06    actual   
6   ch

In [69]:
# 4. AGGREGATE DAILY FORECASTS TO MONTHLY GRAIN
# This creates a CSV with monthly aggregated forecast values

log("")
log_section("AGGREGATING DAILY FORECASTS TO MONTHLY GRAIN")

# Add year-month column for grouping
df_all_forecasts['year_month'] = df_all_forecasts['forecast_date'].dt.to_period('M')

# Group by time series identifiers and year-month, then aggregate
monthly_forecasts = df_all_forecasts.groupby([
    'metric', 
    'grouping', 
    'group_key', 
    'model', 
    'year_month'
]).agg({
    'forecast_value': 'mean',  # Average daily values for the month (backward compatibility)
    'forecast_p50': 'mean',    # Average P50 point estimates for the month
    'forecast_p10': 'mean',    # Average P10 lower bounds for the month
    'forecast_p90': 'mean',    # Average P90 upper bounds for the month
    'forecast_date': ['min', 'max'],  # First and last date in month
    'last_historical_date': 'first',
    'forecast_horizon_days': ['min', 'max']  # Min and max horizon days in month
}).reset_index()

# Flatten column names
monthly_forecasts.columns = [
    'metric', 'grouping', 'group_key', 'model', 'year_month',
    'avg_forecast_value',
    'avg_forecast_p50', 
    'avg_forecast_p10', 
    'avg_forecast_p90',
    'month_start_date', 'month_end_date',
    'last_historical_date',
    'forecast_horizon_days_min', 'forecast_horizon_days_max'
]

# Convert year_month back to string for better CSV readability
monthly_forecasts['year_month'] = monthly_forecasts['year_month'].astype(str)

# Add a column for number of days in the forecast month period
monthly_forecasts['days_in_month_period'] = (
    pd.to_datetime(monthly_forecasts['month_end_date']) - 
    pd.to_datetime(monthly_forecasts['month_start_date'])
).dt.days + 1

log(f"\nMonthly Forecast Summary:")
log(f"  Total rows: {len(monthly_forecasts):,}")
log(f"  Unique time series: {len(monthly_forecasts.groupby(['metric', 'grouping', 'group_key']))}")
log(f"  Unique metrics: {monthly_forecasts['metric'].nunique()}")
log(f"  Date range: {monthly_forecasts['year_month'].min()} to {monthly_forecasts['year_month'].max()}")

log(f"\nSample monthly data:")
log(monthly_forecasts.head(10))

# Save to CSV
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# Export to CoreWeave Object Storage (CSV + XLSX)
save_df_to_s3('forecast_monthly_values', monthly_forecasts)
log(f"\n✓ Saved {len(monthly_forecasts):,} monthly forecast values")

# Show utilization metrics at monthly grain
log("")
log_section("UTILIZATION METRICS - MONTHLY AGGREGATION")

util_monthly = monthly_forecasts[monthly_forecasts['metric'].str.contains('util', case=False)]
if len(util_monthly) > 0:
    log(f"\nMonthly utilization forecasts:")
    log(f"  Rows: {len(util_monthly):,}")
    
    util_monthly_summary = util_monthly.groupby(['metric', 'grouping', 'group_key']).agg({
        'avg_forecast_p50': ['min', 'max', 'mean'],
        'avg_forecast_p10': ['min', 'max', 'mean'],
        'avg_forecast_p90': ['min', 'max', 'mean']
    }).round(4)
    
    log(f"\nUtilization monthly summary (by time series):")
    log(util_monthly_summary.head(20))
    
    # Bounds check for P50, P10, and P90
    max_p50 = util_monthly['avg_forecast_p50'].max()
    min_p50 = util_monthly['avg_forecast_p50'].min()
    max_p10 = util_monthly['avg_forecast_p10'].max()
    min_p10 = util_monthly['avg_forecast_p10'].min()
    max_p90 = util_monthly['avg_forecast_p90'].max()
    min_p90 = util_monthly['avg_forecast_p90'].min()
    
    log("")
    log_section("MONTHLY BOUNDS CHECK")
    log(f"P50 (point estimate) range: [{min_p50:.6f}, {max_p50:.6f}]")
    log(f"P10 (lower bound) range:    [{min_p10:.6f}, {max_p10:.6f}]")
    log(f"P90 (upper bound) range:    [{min_p90:.6f}, {max_p90:.6f}]")
    
    # Check P50 bounds
    if max_p50 > 1.0:
        exceeds = util_monthly[util_monthly['avg_forecast_p50'] > 1.0]
        log(f"⚠️  WARNING: {len(exceeds):,} monthly P50 values exceed 1.0")
    else:
        log(f"✓ All monthly P50 utilization forecasts within [0, 1] bounds")
    
    if min_p50 < 0.0:
        below = util_monthly[util_monthly['avg_forecast_p50'] < 0.0]
        log(f"⚠️  WARNING: {len(below):,} monthly P50 values below 0.0")
    else:
        log(f"✓ All monthly P50 utilization forecasts >= 0.0")
    
    # Check P10 bounds
    if max_p10 > 1.0:
        exceeds = util_monthly[util_monthly['avg_forecast_p10'] > 1.0]
        log(f"⚠️  WARNING: {len(exceeds):,} monthly P10 values exceed 1.0")
    else:
        log(f"✓ All monthly P10 utilization forecasts within [0, 1] bounds")
    
    if min_p10 < 0.0:
        below = util_monthly[util_monthly['avg_forecast_p10'] < 0.0]
        log(f"⚠️  WARNING: {len(below):,} monthly P10 values below 0.0")
    else:
        log(f"✓ All monthly P10 utilization forecasts >= 0.0")
    
    # Check P90 bounds
    if max_p90 > 1.0:
        exceeds = util_monthly[util_monthly['avg_forecast_p90'] > 1.0]
        log(f"⚠️  WARNING: {len(exceeds):,} monthly P90 values exceed 1.0")
    else:
        log(f"✓ All monthly P90 utilization forecasts within [0, 1] bounds")
    
    if min_p90 < 0.0:
        below = util_monthly[util_monthly['avg_forecast_p90'] < 0.0]
        log(f"⚠️  WARNING: {len(below):,} monthly P90 values below 0.0")
    else:
        log(f"✓ All monthly P90 utilization forecasts >= 0.0")
else:
    log("No utilization metrics found in monthly forecast data")

log(f"\n{'='*80}")
log(f"✓ Monthly forecast aggregation complete")
log(f"  Output: s3://{S3_BUCKET}/{S3_PREFIX}forecast_monthly_values.csv")
log(f"{'='*80}")



AGGREGATING DAILY FORECASTS TO MONTHLY GRAIN

Monthly Forecast Summary:
  Total rows: 22,896
  Unique time series: 618
  Unique metrics: 10
  Date range: 2026-01 to 2029-06

Sample monthly data:
                 metric grouping group_key         model year_month  \
0  chip_power_fleet_p50      All       All  holt_winters    2026-01   
1  chip_power_fleet_p50      All       All  holt_winters    2026-02   
2  chip_power_fleet_p50      All       All  holt_winters    2026-03   
3  chip_power_fleet_p50      All       All  holt_winters    2026-04   
4  chip_power_fleet_p50      All       All  holt_winters    2026-05   
5  chip_power_fleet_p50      All       All  holt_winters    2026-06   
6  chip_power_fleet_p50      All       All  holt_winters    2026-07   
7  chip_power_fleet_p50      All       All  holt_winters    2026-08   
8  chip_power_fleet_p50      All       All  holt_winters    2026-09   
9  chip_power_fleet_p50      All       All  holt_winters    2026-10   

   avg_forecast_value 

In [70]:
# Summary Statistics
log("")
log_section("SUMMARY STATISTICS")

log(f"\nTotal model runs: {len(df_all_models)}")
log(f"Total best models selected: {len(df_best_models)}")
log(f"Total plots generated: {len(all_plots)}")

log("\n--- Best Model Distribution ---")
log(df_best_models['model'].value_counts())

log("\n--- Average Error Metrics by Model (Best Models Only) ---")
log(df_best_models.groupby('model')[['MSE', 'RMSE', 'MAPE']].mean())

log("\n--- Best Models by Metric ---")
for metric in METRICS:
    metric_best = df_best_models[df_best_models['metric'] == metric]
    if len(metric_best) > 0:
        log(f"\n{metric}:")
        log(f"  Most common best model: {metric_best['model'].mode().values[0] if len(metric_best['model'].mode()) > 0 else 'N/A'}")
        log(f"  Avg RMSE: {metric_best['RMSE'].mean():.4f}")
        log(f"  Avg MAPE: {metric_best['MAPE'].mean():.2f}%")

log("")
log_section("FILES GENERATED (CoreWeave Object Storage)")
print_s3_manifest()
log("="*80)


SUMMARY STATISTICS

Total model runs: 2225
Total best models selected: 618
Total plots generated: 618

--- Best Model Distribution ---
model
arima                    221
exponential_smoothing    183
sarima                   117
holt_winters              97
Name: count, dtype: int64

--- Average Error Metrics by Model (Best Models Only) ---
                                MSE           RMSE       MAPE
model                                                        
arima                  1.747326e+11  116004.463548  26.263838
exponential_smoothing  3.549569e+10   61624.140359  17.474783
holt_winters           5.110172e+10  103537.627625  22.336293
sarima                 6.750924e+09   39178.040301  24.758241

--- Best Models by Metric ---

gpu_util_p50:
  Most common best model: arima
  Avg RMSE: 0.0584
  Avg MAPE: 21.57%

gpu_util_p95:
  Most common best model: arima
  Avg RMSE: 0.0620
  Avg MAPE: 13.93%

tensor_util_p50:
  Most common best model: arima
  Avg RMSE: 0.0889
  Avg MAPE: 29.

In [71]:
# PERFORMANCE ANALYSIS
log("")
log_section("PERFORMANCE ANALYSIS")

# Calculate performance metrics
total_model_runs = len(df_all_models)
total_time_series = len(df_best_models)
avg_time_per_series = forecast_time / total_time_series if total_time_series > 0 else 0

log(f"\nThroughput Metrics:")
log(f"  Total model runs: {total_model_runs}")
log(f"  Unique time series: {total_time_series}")
log(f"  Total time: {forecast_time:.2f}s ({forecast_time/60:.2f}m)")
log(f"  Time per series: {avg_time_per_series:.2f}s")
log(f"  Series per second: {total_time_series/forecast_time:.2f}")

# Estimate sequential time
sequential_time = forecast_time * CONFIG['n_workers']
speedup = sequential_time / forecast_time if forecast_time > 0 else 0

log(f"\nParallel Efficiency:")
log(f"  Workers used: {CONFIG['n_workers']}")
log(f"  Estimated sequential time: {sequential_time/60:.1f} minutes")
log(f"  Actual parallel time: {forecast_time/60:.1f} minutes")
log(f"  Speedup: {speedup:.1f}x")
log(f"  Parallel efficiency: {(speedup/CONFIG['n_workers']*100):.1f}%")

log(f"\nResource Utilization:")
log(f"  CPU cores: 32 available, {CONFIG['n_workers']} used ({CONFIG['n_workers']/32*100:.0f}%)")
log(f"  RAM: 512GB available")
log(f"  Spark cluster: Available but not used (multiprocessing sufficient for this dataset)")

log("")
log_section("OPTIMIZATION RECOMMENDATIONS")

if total_time_series < 100:
    log("\n✓ Dataset size: SMALL (< 100 time series)")
    log("  Recommendation: Current parallel processing is optimal")
    log("  Alternative: Could use sequential processing if needed")
    
elif total_time_series < 500:
    log("\n✓ Dataset size: MEDIUM (100-500 time series)")
    log("  Recommendation: Standard parallel processing (current method) is optimal")
    log("  Workers: 30 is good, could increase to 31 for marginal gains")
    
elif total_time_series < 2000:
    log("\n⚡ Dataset size: LARGE (500-2000 time series)")
    log("  Recommendation: Consider chunked processing for better memory management")
    log("  Set: USE_CHUNKED = True, chunk_size = 150")
    
else:
    log("\n⚡⚡ Dataset size: VERY LARGE (> 2000 time series)")
    log("  Recommendation: Use chunked processing")
    log("  Set: USE_CHUNKED = True, chunk_size = 100-200")
    log("  Consider: Spark distributed processing for > 5000 time series")

log(f"\n{'='*80}")


PERFORMANCE ANALYSIS

Throughput Metrics:
  Total model runs: 2225
  Unique time series: 618
  Total time: 30.44s (0.51m)
  Time per series: 0.05s
  Series per second: 20.30

Parallel Efficiency:
  Workers used: 30
  Estimated sequential time: 15.2 minutes
  Actual parallel time: 0.5 minutes
  Speedup: 30.0x
  Parallel efficiency: 100.0%

Resource Utilization:
  CPU cores: 32 available, 30 used (94%)
  RAM: 512GB available
  Spark cluster: Available but not used (multiprocessing sufficient for this dataset)

OPTIMIZATION RECOMMENDATIONS

⚡ Dataset size: LARGE (500-2000 time series)
  Recommendation: Consider chunked processing for better memory management
  Set: USE_CHUNKED = True, chunk_size = 150



In [72]:
# For Kubeflow, you need to use the file browser
print("To download from Kubeflow Notebooks:")
print("=" * 80)
print("\n1. Look at the LEFT SIDEBAR in JupyterLab")
print("2. Click the FOLDER icon (File Browser)")
print("3. Find 'forecasts.zip' in the file list")
print("4. RIGHT-CLICK on 'forecasts.zip'")
print("5. Select 'Download'")
print("\nAlternatively:")
print("- Go to the URL: /files/forecasts.zip")
print("- Or click on the file and use the Download button in the toolbar")
print("\n" + "=" * 80)
print("\nFile ready to download:")
print("  forecasts.zip (20 MB - contains time_series_forecasts.html)")

To download from Kubeflow Notebooks:

1. Look at the LEFT SIDEBAR in JupyterLab
2. Click the FOLDER icon (File Browser)
3. Find 'forecasts.zip' in the file list
4. RIGHT-CLICK on 'forecasts.zip'
5. Select 'Download'

Alternatively:
- Go to the URL: /files/forecasts.zip
- Or click on the file and use the Download button in the toolbar


File ready to download:
  forecasts.zip (20 MB - contains time_series_forecasts.html)


# 🌳 Hierarchical Forecasting (v1 — Patch 1: scaffolding + one-metric POC)

**Design note — embedded.** This section adds a *daily-grain* hierarchical
forecasting framework on top of the existing SparkCaster executor flow. It is
**additive**: nothing above this section changes, so restart-and-run-all still
reproduces the original outputs. Patch 2 will make this the main path; patch 3
completes validation/exports/cleanup.

## Why hierarchy (and why the current cells aren't one)
The existing `create_time_series_tasks` builds **8 independent averaged trees**
— it forecasts `All`, `region_summary`, `product_resolved`, … each by an
*unweighted* `avg(metric)` computed separately. Those worlds are mutually
inconsistent: `avg(All) ≠` any exposure-weighted combination of the product- or
region-level forecasts. This section replaces that with **one shared
operational base hierarchy** and derives every reporting cut from it.

## One shared base grain
```
BASE = (region_summary, product_resolved, customer_segment)
```
Every reporting cut is a **subset of the base dims**, produced by aggregating
the base back up — never a second independent tree:

| Reporting cut | Derivation from base |
|---|---|
| `All` | aggregate over all base dims |
| `region_summary` | aggregate over product_resolved, customer_segment |
| `product_resolved` | aggregate over region_summary, customer_segment |
| `customer_segment` | aggregate over region_summary, product_resolved |
| `product_segment` | map product_resolved→segment, then aggregate |
| `region_summary+product_resolved` | aggregate over customer_segment |
| `region_summary+product_segment` | map→segment, aggregate over customer_segment |

`product_segment` is **derived from `product_resolved`** via a deterministic
map built from the source. Non-deterministic products (a product seen under >1
segment) are resolved to the **most frequent** segment and **logged as
exceptions** — see structure validation.

## Metric handling
- **Non-additive percentile metrics** (`gpu_util_p50`, `tensor_util_p95`,
  `chip_power_p50`, `redfish_power_p95`, …) — i.e. *all current `METRICS`* —
  use the **weighted hierarchical approximation**: forecast at the base
  (working) level, then roll up with
  **`weight = node_count_daily_avg × gpu_count_expected`**.
  Parent = `Σ(child·weight)/Σ(weight)`. This is an **approximate coherent
  rollup**, *not* a true percentile reconciliation — a weighted average of p50s
  is not a p50. This is stated in code and in the caveats cell.
- **Additive / ratio-reconstructable metrics** — forecast an anchor level,
  estimate smoothed time-varying child **shares** (non-negative, sum-to-1 per
  split per day), allocate top-down, recompute rollups. The share framework is
  scaffolded here (`smoothed_shares`, with an equal-split fallback) and becomes
  the main path in patch 2, because **no current `METRICS` entry is truly
  additive** — so v1's POC exercises the weighted-percentile path end-to-end.

## Flow (preserves the executor path)
1. Build a **hierarchy-ready daily base fact** from the source table
   (`build_base_fact`) — weighted-mean metric + summed exposure per base node.
2. Forecast **base-node** series through the **unchanged**
   `forecast_time_series_row` executor function (production path, canonical
   schema, real P10/P90 intervals).
3. **Weighted-roll up** base-node daily forecast paths to every reporting cut
   on the driver, emitting results in the **same canonical schema** so plotting
   and export need no changes.
4. Aggregate daily→monthly **after** the daily hierarchy logic.

Exposure weights over the forecast horizon use a **smoothed recent-history
average held flat per base node** (documented v1 fallback). Patch 2 upgrades
this to a forecasted-exposure path.


In [73]:
# ══════════════════════════════════════════════════════════════════════════
# HIERARCHY v1 — CONFIG, COLUMN RESOLUTION, SEGMENT MAP, STRUCTURE VALIDATION
# (pure driver-side; the rollup/share math below is unit-tested offline)
# ══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

log_section("HIERARCHY v1 — STRUCTURE SETUP & VALIDATION")

# ── One shared operational base grain ─────────────────────────────────────
BASE_DIMS = ['region_summary', 'product_resolved', 'customer_segment']

# Reporting cuts: each is a SUBSET of base dims (single shared tree). The
# derived dim 'product_segment' is produced from product_resolved (see map).
REPORTING_CUTS = {
    'All':                             [],
    'region_summary':                  ['region_summary'],
    'product_resolved':                ['product_resolved'],
    'customer_segment':                ['customer_segment'],
    'product_segment':                 ['product_segment'],
    'region_summary+product_resolved': ['region_summary', 'product_resolved'],
    'region_summary+product_segment':  ['region_summary', 'product_segment'],
}

def is_util_metric(name):
    return any(k in name.lower() for k in ('util', 'utilization', 'usage', 'saturation'))

def clip_metric(a, util):
    a = np.maximum(np.asarray(a, dtype=float), 0.0)      # metrics are non-negative
    return np.minimum(a, 1.0) if util else a             # util bounded to [0,1]

# ── Resolve physical column names (source has drifted: plain vs *_norm) ─────
def _resolve(cols, *cands):
    for c in cands:
        if c in cols:
            return c
    return None

_cols = set(df.columns)
COLMAP = {
    'region_summary':   'region_summary',   # created upstream in the prep cell
    'product_resolved': _resolve(_cols, 'product_resolved', 'product_resolved'),
    'customer_segment': _resolve(_cols, 'customer_segment', 'customer_segment'),
    'product_segment':  _resolve(_cols, 'product_segment', 'product_segment'),
    'node_count':       _resolve(_cols, 'node_count_daily_avg', 'node_count', 'record_count'),
    'gpu_count':        _resolve(_cols, 'gpu_count_expected', 'gpu_count'),
    'day':              _resolve(_cols, 'day'),
}
log("Resolved hierarchy columns → physical source columns:")
for k, v in COLMAP.items():
    log(f"    {k:18} -> {v}")

_missing_dims = [k for k in ('product_resolved', 'customer_segment', 'day') if COLMAP[k] is None]
assert not _missing_dims, f"STRUCTURE FAIL: required base columns missing from source: {_missing_dims}"

# Exposure-weight column availability drives the documented fallback ladder.
EXPOSURE_MODE = (
    'node_x_gpu' if COLMAP['node_count'] and COLMAP['gpu_count'] else
    'node_only'  if COLMAP['node_count'] else
    'gpu_only'   if COLMAP['gpu_count'] else
    'unweighted')
log(f"\nExposure weight mode: {EXPOSURE_MODE}  "
    f"(target = node_count_daily_avg × gpu_count_expected)")
if EXPOSURE_MODE != 'node_x_gpu':
    log("⚠️  FALLBACK: full node×gpu exposure not available in source; using the "
        f"'{EXPOSURE_MODE}' proxy. Rollups are still coherent under this weight, "
        "but document this in planning outputs.")

# ── product_segment derivation map (deterministic + logged exceptions) ──────
def build_segment_map(pairs):
    from collections import defaultdict, Counter
    seen = defaultdict(Counter)
    for pr, seg in pairs:
        if pr is None:
            continue
        seen[pr][seg if seg is not None else '∅'] += 1
    mapping, exceptions = {}, []
    for pr, counter in seen.items():
        if len(counter) == 1:
            mapping[pr] = next(iter(counter))
        else:
            chosen = counter.most_common(1)[0][0]          # stable fallback: modal segment
            mapping[pr] = chosen
            exceptions.append({'product_resolved': pr, 'segments': dict(counter),
                               'chosen': chosen})
    return mapping, exceptions

_pr, _seg = COLMAP['product_resolved'], COLMAP['product_segment']
if _seg is not None:
    _pairs = (df.select(_pr, _seg).distinct().toPandas()
              .itertuples(index=False, name=None))
    SEGMENT_MAP, SEGMENT_EXCEPTIONS = build_segment_map(_pairs)
else:
    SEGMENT_MAP, SEGMENT_EXCEPTIONS = {}, []
    log("⚠️  No product_segment column in source; 'product_segment' cuts will be "
        "skipped in v1 until an upstream mapping is provided.")

log(f"\nproduct_segment derivation: {len(SEGMENT_MAP)} products mapped, "
    f"{len(SEGMENT_EXCEPTIONS)} non-deterministic exception(s).")
for e in SEGMENT_EXCEPTIONS:
    log(f"    ⚠️ {e['product_resolved']}: seen as {e['segments']} → chose '{e['chosen']}'")

# ── STRUCTURE VALIDATION ────────────────────────────────────────────────────
_structure_ok = True

# (1) every required cut derivable from the shared base (subset-of-base check)
for cut, dims in REPORTING_CUTS.items():
    for d in dims:
        derivable = (d in BASE_DIMS) or (d == 'product_segment' and _seg is not None)
        if not derivable:
            log(f"❌ cut '{cut}' needs dim '{d}' not derivable from base"); _structure_ok = False
log("✓ (1) all reporting cuts derivable from the shared base"
    if _structure_ok else "❌ (1) some cuts not derivable")

# (2) product_segment derivation deterministic OR exceptions documented
log(f"✓ (2) product_segment map deterministic ({len(SEGMENT_EXCEPTIONS)} exception(s) "
    "documented above, resolved to modal segment)")

# (3) no duplicate/conflicting path: one product_resolved → exactly one segment
_dupe = [pr for pr, s in SEGMENT_MAP.items() if not isinstance(s, str)]
assert not _dupe, f"STRUCTURE FAIL: ambiguous segment mapping for {_dupe}"
log("✓ (3) no product maps to conflicting segments (single shared tree preserved)")

assert _structure_ok, "STRUCTURE VALIDATION FAILED — see messages above"
log("\n✅ STRUCTURE VALIDATION PASSED")


HIERARCHY v1 — STRUCTURE SETUP & VALIDATION
Resolved hierarchy columns → physical source columns:
    region_summary     -> region_summary
    product_resolved   -> product_resolved
    customer_segment   -> customer_segment
    product_segment    -> product_segment
    node_count         -> node_count_daily_avg
    gpu_count          -> gpu_count_expected
    day                -> day

Exposure weight mode: node_x_gpu  (target = node_count_daily_avg × gpu_count_expected)



product_segment derivation: 13 products mapped, 0 non-deterministic exception(s).
✓ (1) all reporting cuts derivable from the shared base
✓ (2) product_segment map deterministic (0 exception(s) documented above, resolved to modal segment)
✓ (3) no product maps to conflicting segments (single shared tree preserved)

✅ STRUCTURE VALIDATION PASSED


In [74]:
# ══════════════════════════════════════════════════════════════════════════
# HIERARCHY v1 — HIERARCHY-READY DAILY BASE FACT  (built from the daily source,
# NOT the 15-min raw table) + DATA VALIDATION
# ══════════════════════════════════════════════════════════════════════════
from functools import reduce
from pyspark.sql import functions as F

log_section("HIERARCHY v1 — BASE FACT BUILDER & DATA VALIDATION")

def _exposure_col():
    """Row-level exposure per the resolved fallback ladder (>= 0, null→0)."""
    nc = F.coalesce(F.col(COLMAP['node_count']), F.lit(0.0)) if COLMAP['node_count'] else None
    gc = F.coalesce(F.col(COLMAP['gpu_count']),  F.lit(0.0)) if COLMAP['gpu_count']  else None
    if EXPOSURE_MODE == 'node_x_gpu':  expo = nc * gc
    elif EXPOSURE_MODE == 'node_only': expo = nc
    elif EXPOSURE_MODE == 'gpu_only':  expo = gc
    else:                              expo = F.lit(1.0)          # unweighted fallback
    return F.greatest(expo, F.lit(0.0))                          # enforce non-negative

def _segment_col():
    """Map product_resolved -> product_segment on the driver-built SEGMENT_MAP."""
    pr = F.col(COLMAP['product_resolved'])
    if not SEGMENT_MAP:
        return F.lit(None).cast('string')
    # build the product_resolved -> product_segment CASE chain
    col = None
    for k, v in SEGMENT_MAP.items():
        col = F.when(pr == k, F.lit(v)) if col is None else col.when(pr == k, F.lit(v))
    return col.otherwise(F.lit('UNKNOWN'))

def build_base_fact(metric):
    """Aggregate the daily source to the shared base grain for one metric.
    Returns Spark DF: BASE_DIMS + product_segment + day + value + weight
      value  = exposure-weighted mean of the (already-percentile) source metric
      weight = summed exposure (used for coherent rollups & horizon weighting)."""
    assert metric in df.columns, f"metric '{metric}' not in source columns"
    d = (df
         .withColumn('_expo', _exposure_col())
         .withColumn('product_segment', _segment_col())
         .withColumn('_m', F.col(metric).cast('double')))
    # weighted numerator; keep weight-with-nonnull-metric separate so a null
    # metric doesn't inflate the denominator
    d = d.withColumn('_wv', F.when(F.col('_m').isNotNull(), F.col('_m') * F.col('_expo')))
    d = d.withColumn('_w_eff', F.when(F.col('_m').isNotNull(), F.col('_expo')))
    grp = BASE_DIMS + ['product_segment', 'day']
    agg = (d.groupBy(*grp).agg(
              F.sum('_wv').alias('_wv'),
              F.sum('_w_eff').alias('_w_eff'),
              F.sum('_expo').alias('weight'),
              F.avg('_m').alias('_mean_v'))
           .withColumn('value', F.when(F.col('_w_eff') > 0, F.col('_wv') / F.col('_w_eff'))
                                  .otherwise(F.col('_mean_v')))   # zero-weight fallback
           .select(*BASE_DIMS, 'product_segment', 'day', 'value', 'weight'))
    return agg

def validate_base_fact(base_sdf, metric, min_days=30):
    """DATA VALIDATION on the base fact for one metric. Returns dict of stats;
    asserts hard failures (missing keys, no usable history)."""
    log_section(f"DATA VALIDATION — base fact for '{metric}'")
    n = base_sdf.count()
    util = is_util_metric(metric)

    # null rates & cardinality for hierarchy dims + weight
    exprs = []
    for c in BASE_DIMS + ['product_segment', 'value', 'weight']:
        exprs += [F.avg(F.col(c).isNull().cast('double')).alias(f'{c}__nullrate')]
    for c in BASE_DIMS + ['product_segment']:
        exprs += [F.countDistinct(c).alias(f'{c}__ndistinct')]
    stats = base_sdf.select(*exprs).first().asDict()

    log(f"  rows at base grain: {n:,}")
    for c in BASE_DIMS + ['product_segment']:
        log(f"    {c:18} null={stats[f'{c}__nullrate']:.3%}  cardinality={stats[f'{c}__ndistinct']}")
    log(f"    {'value':18} null={stats['value__nullrate']:.3%}")
    log(f"    {'weight':18} null={stats['weight__nullrate']:.3%}")

    # exposure weight: non-negative + how often null/zero
    wst = base_sdf.select(
        F.min('weight').alias('wmin'),
        F.avg((F.col('weight') == 0).cast('double')).alias('zero_rate'),
        F.avg(F.col('weight').isNull().cast('double')).alias('null_rate')).first()
    log(f"    weight: min={wst['wmin']}  zero_rate={wst['zero_rate']:.3%}  "
        f"null_rate={wst['null_rate']:.3%}  (exposure must be >= 0)")
    assert (wst['wmin'] is None) or (wst['wmin'] >= 0), "DATA FAIL: negative exposure weight"

    # enough daily history at base level to fit the models (executor needs >=21)
    per_node = (base_sdf.groupBy(*BASE_DIMS)
                .agg(F.countDistinct('day').alias('ndays')))
    hd = per_node.select(F.min('ndays').alias('mn'), F.max('ndays').alias('mx'),
                         F.avg('ndays').alias('av'),
                         F.avg((F.col('ndays') >= 21).cast('double')).alias('fit_frac')).first()
    log(f"    base-node history (days): min={hd['mn']} avg={hd['av']:.0f} max={hd['mx']}  "
        f"fraction with >=21d (fittable): {hd['fit_frac']:.1%}")
    assert hd['mx'] is not None and hd['mx'] >= min_days, \
        f"DATA FAIL: longest base-node history {hd['mx']}d < {min_days}d required"

    # complete keys for chosen dims
    for c in BASE_DIMS:
        assert stats[f'{c}__nullrate'] < 1.0, f"DATA FAIL: dim '{c}' entirely null"
    log("✅ DATA VALIDATION PASSED (keys present, exposure >=0, sufficient history)")
    return {'rows': n, **stats, **{k: wst[k] for k in wst.asDict()}}

log("✓ build_base_fact / validate_base_fact ready")


HIERARCHY v1 — BASE FACT BUILDER & DATA VALIDATION
✓ build_base_fact / validate_base_fact ready


In [75]:
# ══════════════════════════════════════════════════════════════════════════
# HIERARCHY v1 — ONE-METRIC PROOF OF CONCEPT
# Forecast BASE-NODE series through the UNCHANGED executor path, then capture
# per-node horizon exposure weights for the rollup.
# ══════════════════════════════════════════════════════════════════════════
import json, time
from pyspark.sql import functions as F

# Pick a representative metric (prefer a bounded util metric) present in source.
_candidates = ['gpu_util_p50'] + [m for m in METRICS if m != 'gpu_util_p50']
POC_METRIC = next((m for m in _candidates if m in df.columns), None)
assert POC_METRIC is not None, f"none of METRICS present in source: {METRICS}"
log_section(f"HIERARCHY v1 — POC on '{POC_METRIC}'  (util={is_util_metric(POC_METRIC)})")

# ── 1. Build + validate the base fact ───────────────────────────────────────
base_fact = build_base_fact(POC_METRIC).cache()
_ = validate_base_fact(base_fact, POC_METRIC)

# ── 2. Base-node tasks (one series per base node) → executor path ────────────
BASE_SEP = '||'
def create_base_tasks(base_sdf, metric):
    return (base_sdf
            .withColumn('group_key', F.concat_ws(BASE_SEP, *[F.col(c) for c in BASE_DIMS]))
            .groupBy('group_key')
            .agg(F.collect_list(F.struct('day', 'value')).alias('time_series_data'))
            .withColumn('metric', F.lit(metric))
            .withColumn('grouping_name', F.lit('__base__'))
            .select('metric', 'grouping_name', 'group_key', 'time_series_data'))

log("Step 1: creating base-node tasks…")
base_tasks = create_base_tasks(base_fact, POC_METRIC)
n_base = base_tasks.count()
log(f"  {n_base} base-node series")

log("Step 2: distributed forecasting on base nodes (reusing forecast_time_series_row)…")
_t = time.time()
base_results = [json.loads(r) for r in
                base_tasks.repartition(min(n_base, 200)).rdd.map(forecast_time_series_row).collect()]
log(f"  done in {time.time()-_t:.1f}s  "
    f"(completed={sum(r.get('status')=='completed' for r in base_results)}, "
    f"skipped={sum(r.get('status')=='skipped' for r in base_results)}, "
    f"error={sum(r.get('status')=='error' for r in base_results)})")

# ── 3. Horizon exposure weight per base node ────────────────────────────────
# v1 FALLBACK: smoothed recent-history mean exposure, held FLAT over the
# horizon (documented). Patch 2 replaces this with a forecasted-exposure path.
_RECENT_DAYS = 60
_maxday = base_fact.select(F.max('day')).first()[0]
_recent = base_fact.filter(F.col('day') >= F.date_sub(F.lit(_maxday), _RECENT_DAYS))
_bw = (_recent
       .withColumn('group_key', F.concat_ws(BASE_SEP, *[F.col(c) for c in BASE_DIMS]))
       .groupBy('group_key', *BASE_DIMS, 'product_segment')
       .agg(F.avg('weight').alias('w')).toPandas())
BASE_WEIGHTS = _bw.set_index('group_key')
log(f"Step 3: captured flat horizon exposure for {len(BASE_WEIGHTS)} base nodes "
    f"(smoothed over last {_RECENT_DAYS}d).")
log(f"\n✓ POC base forecasts + weights ready for rollup "
    f"({sum(r.get('status')=='completed' for r in base_results)} usable nodes)")


HIERARCHY v1 — POC on 'gpu_util_p50'  (util=True)
DATA VALIDATION — base fact for 'gpu_util_p50'
  rows at base grain: 17,695
    region_summary     null=0.000%  cardinality=2
    product_resolved   null=0.000%  cardinality=13
    customer_segment   null=0.000%  cardinality=4
    product_segment    null=0.000%  cardinality=4
    value              null=6.657%
    weight             null=0.000%
    weight: min=0.0  zero_rate=6.657%  null_rate=0.000%  (exposure must be >= 0)
    base-node history (days): min=17 avg=340 max=517  fraction with >=21d (fittable): 96.2%
✅ DATA VALIDATION PASSED (keys present, exposure >=0, sufficient history)
Step 1: creating base-node tasks…
  52 base-node series
Step 2: distributed forecasting on base nodes (reusing forecast_time_series_row)…
  done in 7.0s  (completed=50, skipped=2, error=0)
Step 3: captured flat horizon exposure for 46 base nodes (smoothed over last 60d).

✓ POC base forecasts + weights ready for rollup (50 usable nodes)


In [76]:
# ══════════════════════════════════════════════════════════════════════════
# HIERARCHY v1 — WEIGHTED ROLLUP TO REPORTING CUTS  + APPROXIMATION / INTERVAL /
# MONTHLY-CONSISTENCY VALIDATION  (non-additive percentile path)
#
# APPROXIMATION NOTE: parent = Σ(child·weight)/Σ(weight). This is a WEIGHTED
# HIERARCHICAL APPROXIMATION — a weighted average of p50/p95 is NOT itself a
# true percentile. These are approximate-coherence checks, NOT quantile
# reconciliation checks.
# ══════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd
log_section("HIERARCHY v1 — WEIGHTED ROLLUP + APPROXIMATION VALIDATION")

_UTIL = is_util_metric(POC_METRIC)

# ── weighted rollup (mirrors the offline-tested hier_core.weighted_rollup) ──
# parent = Σ(child·weight)/Σ(weight); zero-weight group/day → unweighted mean.
def weighted_rollup(child_df, cut_dims, valcol, daycol='forecast_date'):
    cols = list(dict.fromkeys(cut_dims + [daycol, 'weight', valcol]))
    d = child_df[cols].copy()
    d['wv'] = d[valcol] * d['weight']
    gb = cut_dims + [daycol]
    agg = d.groupby(gb, dropna=False).agg(wv=('wv', 'sum'), wsum=('weight', 'sum'),
                                          mean_v=(valcol, 'mean')).reset_index()
    agg[valcol] = np.where(agg['wsum'] > 0, agg['wv'] / agg['wsum'], agg['mean_v'])
    return agg[gb + [valcol, 'wsum']].rename(columns={'wsum': 'weight'})

# ── 1. Per-base-node daily forecast frame (p10/p50/p90 + dates) ─────────────
_rows = []
for r in base_results:
    if r.get('status') != 'completed' or not r.get('best_model'):
        continue
    bm = r['best_model']; res = r['results'][bm]
    fc = np.asarray(res.get('forecast', []), float)
    lo = np.asarray(res.get('forecast_lower', []), float)
    hi = np.asarray(res.get('forecast_upper', []), float)
    n = len(fc)
    if n == 0:
        continue
    if len(lo) != n: lo = np.full(n, np.nan)
    if len(hi) != n: hi = np.full(n, np.nan)
    last_hist = pd.to_datetime(list(r.get('train_dates', [])) + list(r.get('test_dates', []))).max()
    dates = pd.date_range(last_hist + pd.Timedelta(days=1), periods=n, freq='D')
    gk = r['group_key']
    _rows.append(pd.DataFrame({'group_key': gk, 'model': bm, 'forecast_date': dates,
                               'horizon_day': np.arange(1, n + 1),
                               'p50': fc, 'p10': lo, 'p90': hi}))
node_fc = pd.concat(_rows, ignore_index=True)

# attach base dims + horizon exposure weight
node_fc = node_fc.merge(BASE_WEIGHTS.reset_index()[['group_key', *BASE_DIMS,
                        'product_segment', 'w']], on='group_key', how='left')
node_fc = node_fc.rename(columns={'w': 'weight'})
node_fc['weight'] = node_fc['weight'].fillna(0.0)
log(f"Base-node forecast frame: {node_fc['group_key'].nunique()} nodes × "
    f"{node_fc['horizon_day'].max()} horizon days = {len(node_fc):,} rows")

# ── 2. Roll up to every reporting cut ───────────────────────────────────────
hier_daily = []
for cut, dims in REPORTING_CUTS.items():
    if any(d == 'product_segment' for d in dims) and not SEGMENT_MAP:
        log(f"  · skip '{cut}' (no product_segment mapping in source)"); continue
    p50 = weighted_rollup(node_fc, dims, 'p50')
    p10 = weighted_rollup(node_fc, dims, 'p10')
    p90 = weighted_rollup(node_fc, dims, 'p90')
    key = dims + ['forecast_date']
    m = p50.merge(p10[key + ['p10']], on=key).merge(p90[key + ['p90']], on=key)
    m['group_key'] = (m[dims].astype(str).agg('|'.join, axis=1) if dims else 'All')
    m['grouping'] = cut
    # bounded-metric clip after rollup
    for c in ('p10', 'p50', 'p90'):
        m[c] = clip_metric(m[c], _UTIL)
    hier_daily.append(m[['grouping', 'group_key', 'forecast_date', 'p10', 'p50', 'p90', 'weight']])
hier_daily_forecasts = pd.concat(hier_daily, ignore_index=True)
log(f"\nRolled up to {hier_daily_forecasts['grouping'].nunique()} cuts, "
    f"{len(hier_daily_forecasts):,} daily rows")

# ══ VALIDATION ══════════════════════════════════════════════════════════════
_ok = True
_TOL = 1e-6

# (a) APPROXIMATION coherence — All computed directly == recomputed via region
_all_direct = weighted_rollup(node_fc, [], 'p50').set_index('forecast_date')['p50']
_region = weighted_rollup(node_fc, ['region_summary'], 'p50')   # carries p50 + weight
_tmp = _region.copy(); _tmp['wv'] = _tmp['p50'] * _tmp['weight']
_all_via = (_tmp.groupby('forecast_date').agg(wv=('wv', 'sum'), w=('weight', 'sum'))
            .assign(p50=lambda x: x.wv / x.w)['p50'])
_j = pd.concat([_all_direct.rename('direct'), _all_via.rename('via_region')], axis=1).dropna()
_mx = (_j['direct'] - _j['via_region']).abs().max()
log(f"(a) rollup coherence  base→All vs region→All  maxdiff={_mx:.2e}  "
    + ("✓" if _mx < _TOL else "❌")); _ok &= _mx < _TOL

# (b) interval ordering p10 <= p50 <= p90 for every row
_bad_iv = ((hier_daily_forecasts['p10'] > hier_daily_forecasts['p50'] + 1e-9) |
           (hier_daily_forecasts['p50'] > hier_daily_forecasts['p90'] + 1e-9)).sum()
log(f"(b) interval ordering p10<=p50<=p90  violations={_bad_iv}  "
    + ("✓" if _bad_iv == 0 else "❌")); _ok &= _bad_iv == 0

# (c) bounded metric range after rollup+intervals
if _UTIL:
    _rng = (hier_daily_forecasts[['p10', 'p50', 'p90']].min().min(),
            hier_daily_forecasts[['p10', 'p50', 'p90']].max().max())
    _inb = _rng[0] >= -1e-9 and _rng[1] <= 1 + 1e-9
    log(f"(c) util range after rollup = [{_rng[0]:.4f}, {_rng[1]:.4f}]  "
        + ("✓ within [0,1]" if _inb else "❌ out of bounds")); _ok &= _inb
else:
    log("(c) non-util metric — only non-negativity enforced ✓")

# (d) horizon length preserved (not silently truncated) per cut
_hz = hier_daily_forecasts.groupby('grouping')['forecast_date'].nunique()
_exp = node_fc['forecast_date'].nunique()
_hz_ok = (_hz == _exp).all()
log(f"(d) horizon days per cut all == {_exp}  " + ("✓" if _hz_ok else f"❌ {_hz.to_dict()}"))
_ok &= _hz_ok

# ── 3. Daily → monthly (AFTER daily rollup) + monthly-consistency check ─────
hd = hier_daily_forecasts.copy()
hd['year_month'] = hd['forecast_date'].dt.to_period('M').astype(str)
hier_monthly_forecasts = (hd.groupby(['grouping', 'group_key', 'year_month'])
                          .agg(avg_p10=('p10', 'mean'), avg_p50=('p50', 'mean'),
                               avg_p90=('p90', 'mean'), days=('forecast_date', 'nunique'))
                          .reset_index())
# (e) monthly == mean(daily) recomputed independently
_chk = (hd.groupby(['grouping', 'group_key', 'year_month'])['p50'].mean().reset_index()
        .merge(hier_monthly_forecasts, on=['grouping', 'group_key', 'year_month']))
_mmx = (_chk['p50'] - _chk['avg_p50']).abs().max()
log(f"(e) monthly == mean(daily) recomputed  maxdiff={_mmx:.2e}  "
    + ("✓" if _mmx < _TOL else "❌")); _ok &= _mmx < _TOL

assert _ok, "HIERARCHY VALIDATION FAILED — see (a)–(e) above"
log("\n✅ APPROXIMATION / INTERVAL / MONTHLY VALIDATION PASSED "
    "(approximate coherence, NOT true quantile reconciliation)")

# ── 4. Persist hierarchy outputs (distinct '_hier' names; existing exports untouched)
try:
    save_df_to_s3(f'forecast_daily_values_hier_{POC_METRIC}', hier_daily_forecasts)
    save_df_to_s3(f'forecast_monthly_values_hier_{POC_METRIC}', hier_monthly_forecasts)
    log(f"✓ saved hierarchy POC outputs for '{POC_METRIC}' (daily + monthly)")
except Exception as _e:
    log(f"(export skipped: {str(_e)[:120]})")

log("\nSample — hierarchy monthly forecast by cut:")
log(hier_monthly_forecasts.groupby('grouping').head(1).to_string(index=False))


HIERARCHY v1 — WEIGHTED ROLLUP + APPROXIMATION VALIDATION
Base-node forecast frame: 50 nodes × 1100 horizon days = 55,000 rows

Rolled up to 7 cuts, 62,434 daily rows
(a) rollup coherence  base→All vs region→All  maxdiff=3.33e-16  ✓
(b) interval ordering p10<=p50<=p90  violations=0  ✓
(c) util range after rollup = [0.0000, 1.0000]  ✓ within [0,1]
(d) horizon days per cut all == 1470  ✓
(e) monthly == mean(daily) recomputed  maxdiff=0.00e+00  ✓

✅ APPROXIMATION / INTERVAL / MONTHLY VALIDATION PASSED (approximate coherence, NOT true quantile reconciliation)
  → s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/20260723_221127/forecast_daily_values_hier_gpu_util_p50.csv
  → s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/20260723_221127/forecast_daily_values_hier_gpu_util_p50.xlsx
  → s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/20260723_221127/forecast_monthly_values_hier_gpu_util_p50.csv
  → s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/20260723_22

## 📋 Hierarchy v1 — assumptions, caveats & approximation rules

**Status: Patch 1 (scaffolding + one-metric POC).** This section is additive and
does not alter the original pipeline above.

### Assumptions
- The daily summary source is the forecasting layer (we do **not** model the
  15-min raw DCGM table). If additive components are later needed, a
  hierarchy-ready daily fact is built upstream with Spark SQL — never per-node
  models on the raw table.
- `region_summary` is `EU` vs `NAM` (from the prep cell). Base grain =
  `(region_summary, product_resolved, customer_segment)`.
- `product_segment` is a **deterministic function of `product_resolved`** built
  from the source; non-deterministic products are resolved to their **modal**
  segment and printed as exceptions in structure validation.
- Column drift is absorbed by `COLMAP` (plain vs `*_norm`); the exact physical
  columns used are printed at setup.

### Approximation rules (be explicit)
- All current `METRICS` are **percentile-style, non-additive**. Their parent /
  intermediate rollups use the **weighted hierarchical approximation**:
  `parent = Σ(child · weight) / Σ(weight)`, `weight = node_count_daily_avg ×
  gpu_count_expected`.
- **A weighted average of p50/p95 is NOT a true percentile.** Validation checks
  (a)/(c) are **approximate-coherence** checks, *not* quantile reconciliation.
- Horizon exposure weight is a **smoothed recent-history mean held flat** per
  base node (v1 fallback). If full `node×gpu` exposure is unavailable, a printed
  fallback proxy (`node_only` / `gpu_only` / `unweighted`) is used.

### Caveats
- P10/P90 rollups are exposure-weighted averages of child intervals — an
  approximation, not a simulated joint interval.
- Zero-weight group/day falls back to an unweighted mean (kept finite).
- The additive **share-allocation** path (`smoothed_shares`, top-down) is
  scaffolded but not exercised, because no current metric is truly additive.

### What passes today (POC, one metric)
Structure ✓ · data ✓ · rollup coherence (base→All == region→All) ✓ · intervals
p10≤p50≤p90 ✓ · bounded util∈[0,1] ✓ · horizon not truncated ✓ · monthly ==
mean(daily) ✓.

### Next
- **Patch 2** — generalize to all metric families; forecast exposure for the
  horizon (replace flat weight); wire hierarchy results into the canonical
  `parsed_results` → `plot_data_list` so existing plots/exports render every cut;
  implement the additive top-down share path for any additive metrics.
- **Patch 3** — full backtest vs the non-hierarchical baseline on a holdout,
  runtime check, export/plot consistency, restart-run-all cleanup.


# 🌳🌳 Hierarchical Forecasting (v1 — Patch 2: generalize + canonicalize)

Patch 2 turns the patch-1 POC into the **main hierarchy path across all metric
families**, and removes the patch-1 flat-weight caveat:

1. **Forecasted exposure (additive top-down).** Exposure
   (`node_count_daily_avg × gpu_count_expected`) is itself an **additive**
   series. We forecast the **anchor total** through the *unchanged* executor
   model, estimate smoothed non-negative shares that **sum to 1** per node, and
   **allocate top-down** — giving a coherent, *time-varying* horizon weight per
   base node (`Σ_node exposure = parent total`, exact by construction). This is
   the additive share-allocation path the design calls for.
2. **All metrics in one distributed pass.** Base-node series for every metric
   are forecast in a **single** Spark job via `forecast_time_series_row` (no
   duplicate forecasting implementation).
3. **Weighted rollup → canonical schema.** Each `(metric, reporting-cut)` is
   emitted as a `parsed_results`-schema entry (`best_model='hierarchical_weighted'`)
   so the **existing** `plot_data_list` builder, plots, and exports consume it
   unchanged. The cell-19 builder is factored into `build_plot_data_list()` so
   baseline and hierarchy share **one** implementation.
4. **Monthly from daily hierarchy.** Monthly outputs are aggregated *after* the
   daily rollup, and validated to equal `mean(daily)`.

Still additive: the original pipeline above is untouched, so restart-run-all
reproduces the baseline, then produces the hierarchy artifacts. Patch 3 will
retrofit cells 19/27/31 to call these shared functions and flip the default
exports after the holdout backtest.


In [77]:
# ══════════════════════════════════════════════════════════════════════════
# HIERARCHY v1 — PATCH 2 SHARED CORE (pure-Python, offline unit-tested)
# Reuses BASE_DIMS / REPORTING_CUTS / TRAIN_SPLIT / is_util_metric / clip_metric
# from the patch-1 cells above.
# ══════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd


def is_util_metric(n): return any(k in n.lower() for k in ('util','utilization','usage','saturation'))
def clip_metric(a, util):
    a = np.maximum(np.asarray(a, float), 0.0); return np.minimum(a, 1.0) if util else a

def _metrics(actual, pred):
    actual = np.asarray(actual, float); pred = np.asarray(pred, float)
    if len(actual) == 0: return {'MSE': np.nan,'RMSE': np.nan,'MAPE': np.nan,'MAE': np.nan}
    mse = float(np.mean((actual-pred)**2)); denom = np.where(actual==0, np.nan, actual)
    return {'MSE': mse, 'RMSE': float(np.sqrt(mse)),
            'MAPE': float(np.nanmean(np.abs((actual-pred)/denom))*100),
            'MAE': float(np.mean(np.abs(actual-pred)))}

# ── weighted rollup (same as tested patch-1 helper, parametrized day column) ──
def weighted_rollup(child_df, cut_dims, valcol, daycol):
    cols = list(dict.fromkeys(cut_dims + [daycol, 'weight', valcol]))
    d = child_df[cols].copy(); d['wv'] = d[valcol]*d['weight']
    gb = cut_dims + [daycol]
    agg = d.groupby(gb, dropna=False).agg(wv=('wv','sum'), wsum=('weight','sum'),
                                          mean_v=(valcol,'mean')).reset_index()
    agg[valcol] = np.where(agg['wsum']>0, agg['wv']/agg['wsum'], agg['mean_v'])
    return agg[gb + [valcol,'weight' if False else 'wsum']].rename(columns={'wsum':'weight'})

# ── ADDITIVE TOP-DOWN: smoothed shares + allocate parent total to base nodes ─
def smoothed_shares_flat(expo_hist, halflife=14, recent=60):
    """expo_hist: [group_key, day, exposure]. Returns flat horizon share per node
    (last EWMA-smoothed share of the daily total), non-negative, summing to 1."""
    d = expo_hist.copy().sort_values('day')
    maxd = d['day'].max(); d = d[d['day'] >= maxd - pd.Timedelta(days=recent)]
    tot = d.groupby('day')['exposure'].transform('sum')
    d['share_raw'] = np.where(tot > 0, d['exposure']/tot, np.nan)
    d = d.groupby('group_key', group_keys=False).apply(
        lambda g: g.assign(share=g.sort_values('day')['share_raw']
                           .ewm(halflife=halflife, ignore_na=True).mean()))
    last = d.sort_values('day').groupby('group_key').tail(1)[['group_key','share']]
    last['share'] = last['share'].clip(lower=0).fillna(0.0)
    s = last['share'].sum()
    last['share'] = last['share']/s if s > 0 else 1.0/len(last)
    return last.reset_index(drop=True)

def allocate_exposure(total_future, shares):
    """total_future: [forecast_date, total]. shares: [group_key, share].
    Returns EXPO_FUTURE [group_key, forecast_date, weight] with
    Σ_node weight(date) == total(date)."""
    tf = total_future.assign(key=1); sh = shares.assign(key=1)
    ef = tf.merge(sh, on='key').drop(columns='key')
    ef['weight'] = ef['total']*ef['share']
    return ef[['group_key','forecast_date','weight']]

# ── canonical entry per (metric, cut) in the cell-19 parsed_results schema ───
def build_cut_entry(metric, cut, dims, node_fc, node_hist, node_testpred, util):
    """node_fc: [group_key,+dims,forecast_date,p10,p50,p90,weight]
       node_hist: [group_key,+dims,day,value,weight]
       node_testpred: [group_key,+dims,day,tp,weight]"""
    key = dims + ['forecast_date']
    p50 = weighted_rollup(node_fc, dims, 'p50', 'forecast_date')
    p10 = weighted_rollup(node_fc, dims, 'p10', 'forecast_date')
    p90 = weighted_rollup(node_fc, dims, 'p90', 'forecast_date')
    fc = p50.merge(p10[key+['p10']], on=key).merge(p90[key+['p90']], on=key).sort_values(key)
    # historical actuals rolled up, then re-split 70/30 at the cut level
    hist = weighted_rollup(node_hist, dims, 'value', 'day').sort_values(dims+['day'])
    # test predictions rolled up (aligned by day)
    tpr = weighted_rollup(node_testpred, dims, 'tp', 'day').sort_values(dims+['day'])

    entries = []
    grpcols = dims if dims else None
    groups = fc.groupby(dims) if dims else [((), fc)]
    for gkey, g in groups:
        gkey = gkey if isinstance(gkey, tuple) else (gkey,)
        sel = {d: v for d, v in zip(dims, gkey)}
        def _f(frame):
            m = frame
            for d, v in sel.items():
                m = m[m[d] == v]
            return m
        gh = _f(hist).sort_values('day'); gt = _f(tpr).sort_values('day'); gf = g.sort_values('forecast_date')
        n = len(gh); split = int(n*TRAIN_SPLIT)
        train, test = gh.iloc[:split], gh.iloc[split:]
        # align test predictions to the cut test window by day
        tp_al = gt[gt['day'].isin(set(test['day']))]
        merged = test.merge(tp_al[['day','tp']], on='day', how='left')
        test_pred = clip_metric(merged['tp'].ffill().fillna(0).to_numpy(), util)
        met = _metrics(test['value'].to_numpy(), test_pred)
        group_key = '|'.join(str(x) for x in gkey) if dims else 'All'
        entries.append({
            'metric': metric, 'grouping': cut, 'grouping_name': cut, 'group_key': group_key,
            'status': 'completed', 'best_model': 'hierarchical_weighted',
            'train_dates': train['day'].astype(str).tolist(),
            'test_dates': test['day'].astype(str).tolist(),
            'train_values': train['value'].tolist(), 'test_values': test['value'].tolist(),
            'results': {'hierarchical_weighted': {
                'status': 'success', 'metrics': met, 'mae': met['MAE'],
                'test_predictions': test_pred.tolist(),
                'forecast': clip_metric(gf['p50'].to_numpy(), util).tolist(),
                'forecast_lower': clip_metric(gf['p10'].to_numpy(), util).tolist(),
                'forecast_upper': clip_metric(gf['p90'].to_numpy(), util).tolist(),
                'fitted': [],
            }}})
    return entries

# ── assemble per-node frames (forecast / history / test-pred) from results ───
def assemble_node_frames(base_results, metric, dims_lookup, expo_future, expo_hist):
    fc_rows, hist_rows, tp_rows = [], [], []
    for r in base_results:
        if r.get('metric') != metric or r.get('status') != 'completed' or not r.get('best_model'):
            continue
        gk = r['group_key']; res = r['results'][r['best_model']]
        fc = np.asarray(res.get('forecast', []), float)
        lo = np.asarray(res.get('forecast_lower', []), float)
        hi = np.asarray(res.get('forecast_upper', []), float)
        n = len(fc)
        if n == 0:
            continue
        if len(lo) != n: lo = np.full(n, np.nan)
        if len(hi) != n: hi = np.full(n, np.nan)
        tr_d = list(r.get('train_dates', [])); te_d = list(r.get('test_dates', []))
        tr_v = list(r.get('train_values', [])); te_v = list(r.get('test_values', []))
        last_hist = pd.to_datetime(tr_d + te_d).max()
        fdates = pd.date_range(last_hist + pd.Timedelta(days=1), periods=n, freq='D')
        fc_rows.append(pd.DataFrame({'group_key': gk, 'forecast_date': fdates,
                                     'p50': fc, 'p10': lo, 'p90': hi}))
        hd = pd.to_datetime(tr_d + te_d); hv = tr_v + te_v
        hist_rows.append(pd.DataFrame({'group_key': gk, 'day': hd, 'value': hv}))
        tp = np.asarray(res.get('test_predictions', []), float)
        if len(tp) == len(te_d) and len(te_d):
            tp_rows.append(pd.DataFrame({'group_key': gk, 'day': pd.to_datetime(te_d), 'tp': tp}))
    node_fc = pd.concat(fc_rows, ignore_index=True)
    node_hist = pd.concat(hist_rows, ignore_index=True)
    node_tp = pd.concat(tp_rows, ignore_index=True) if tp_rows else \
        pd.DataFrame(columns=['group_key', 'day', 'tp'])
    # attach dims
    dl = dims_lookup.reset_index()
    node_fc = node_fc.merge(dl, on='group_key', how='left')
    node_hist = node_hist.merge(dl, on='group_key', how='left')
    node_tp = node_tp.merge(dl, on='group_key', how='left')
    # attach weights (time-varying future exposure; historical exposure)
    node_fc = node_fc.merge(expo_future, on=['group_key', 'forecast_date'], how='left')
    node_fc['weight'] = node_fc['weight'].fillna(0.0)
    node_hist = node_hist.merge(expo_hist.rename(columns={'exposure': 'weight'}),
                                on=['group_key', 'day'], how='left')
    node_hist['weight'] = node_hist['weight'].fillna(0.0)
    node_tp = node_tp.merge(expo_hist.rename(columns={'exposure': 'weight'}),
                            on=['group_key', 'day'], how='left')
    node_tp['weight'] = node_tp['weight'].fillna(0.0)
    return node_fc, node_hist, node_tp

# ── cell-19 builder, factored so baseline + hierarchy share ONE implementation ─
def build_plot_data_list(parsed_results):
    plots = []
    for result in parsed_results:
        if result.get('status') != 'completed':
            continue
        best_model = result.get('best_model')
        grouping = result.get('grouping') or result.get('grouping_name')
        metadata = {'metric': result.get('metric'), 'grouping': grouping,
                    'group_key': result.get('group_key'),
                    'train_dates': pd.to_datetime(result.get('train_dates', [])),
                    'test_dates': pd.to_datetime(result.get('test_dates', [])),
                    'train_values': pd.Series(result.get('train_values', [])),
                    'test_values': pd.Series(result.get('test_values', []))}
        results = {}
        for mn, mr in (result.get('results') or {}).items():
            if mr.get('status') != 'success':
                continue
            results[mn] = {'test_predictions': pd.Series(mr.get('test_predictions', [])),
                'forecast': pd.Series(mr.get('forecast', [])),
                'forecast_lower': pd.Series(mr.get('forecast_lower', [])),
                'forecast_upper': pd.Series(mr.get('forecast_upper', [])),
                'fitted': pd.Series(mr.get('fitted', [])) if mr.get('fitted') is not None else None,
                'metrics': mr.get('metrics', {}), 'metadata': metadata}
        if not results or best_model not in results:
            continue
        plots.append({'metric': result.get('metric'), 'grouping': grouping,
                      'group_key': result.get('group_key'), 'best_model': best_model,
                      'results': results})
    return plots

log('✓ patch-2 shared core ready (exposure allocation, rollup, canonical builder)')


✓ patch-2 shared core ready (exposure allocation, rollup, canonical builder)


In [78]:
# ══════════════════════════════════════════════════════════════════════════
# HIERARCHY v1 (patch 2) — FORECASTED EXPOSURE via ADDITIVE TOP-DOWN
# exposure = node_count_daily_avg × gpu_count_expected  (additive)
# Forecast the anchor total (executor model) → smoothed shares → allocate down.
# Replaces the patch-1 flat horizon weight with a coherent time-varying weight.
# ══════════════════════════════════════════════════════════════════════════
from types import SimpleNamespace
import numpy as np, pandas as pd
from pyspark.sql import functions as F

log_section("HIERARCHY v1 (patch 2) — ADDITIVE EXPOSURE FORECAST")

# ── 1. exposure fact at base grain (uses resolved exposure ladder) ──────────
expo_fact = (df.withColumn('_expo', _exposure_col())
             .withColumn('product_segment', _segment_col())
             .withColumn('group_key', F.concat_ws(BASE_SEP, *[F.col(c) for c in BASE_DIMS]))
             .groupBy('group_key', *BASE_DIMS, 'product_segment', 'day')
             .agg(F.sum('_expo').alias('exposure')))
EXPO_HIST = expo_fact.select('group_key', 'day', 'exposure').toPandas()
EXPO_HIST['day'] = pd.to_datetime(EXPO_HIST['day'])
DIMS_LOOKUP = (expo_fact.select('group_key', *BASE_DIMS, 'product_segment').distinct()
               .toPandas().drop_duplicates('group_key').set_index('group_key'))
log(f"exposure history: {len(EXPO_HIST):,} node-days, {DIMS_LOOKUP.shape[0]} base nodes")

# ── 2. forecast the ANCHOR total-exposure daily series ──────────────────────
# Run through the EXECUTOR path (1-row Spark job), NOT on the driver: the driver
# has no statsmodels (it is shipped to executors via the wheelhouse), so a
# driver-side forecast_time_series_row call would fail on the install path.
anchor = (expo_fact.groupBy('day').agg(F.sum('exposure').alias('value'))
          .orderBy('day').toPandas())
anchor['day'] = pd.to_datetime(anchor['day'])
anchor_last = anchor['day'].max()
_anchor_task = (expo_fact.groupBy('day').agg(F.sum('exposure').alias('value'))
                .groupBy().agg(F.collect_list(F.struct('day', 'value')).alias('time_series_data'))
                .withColumn('metric', F.lit('exposure_total'))
                .withColumn('grouping_name', F.lit('__expo_anchor__'))
                .withColumn('group_key', F.lit('All'))
                .select('metric', 'grouping_name', 'group_key', 'time_series_data'))
_res = json.loads(_anchor_task.rdd.map(forecast_time_series_row).collect()[0])
_bm = _res.get('best_model')
if _res.get('status') == 'completed' and _bm and _bm in _res.get('results', {}):
    _tot = np.maximum(np.asarray(_res['results'][_bm]['forecast'], float), 0.0)  # additive → >=0
    _src = f"model={_bm}"
else:
    # FALLBACK (documented): hold the smoothed recent total flat over the horizon.
    _recent = anchor[anchor['day'] >= anchor_last - pd.Timedelta(days=30)]['value']
    _flat = float(_recent.mean()) if len(_recent) else float(anchor['value'].mean())
    _tot = np.full(int(FORECAST_DAYS), max(_flat, 0.0))
    _src = f"FLAT fallback (anchor fit unavailable: {_res.get('status')}/{_res.get('reason') or _res.get('error','')})"
    log(f"⚠️  anchor exposure forecast fell back to flat total: {_src}")
_fdates = pd.date_range(anchor_last + pd.Timedelta(days=1), periods=len(_tot), freq='D')
total_future = pd.DataFrame({'forecast_date': _fdates, 'total': _tot})
log(f"anchor exposure forecast: {_src}, horizon={len(_tot)}d, "
    f"total range=[{_tot.min():,.0f}, {_tot.max():,.0f}]")

# ── 3. smoothed shares (non-neg, sum-to-1) → allocate top-down ──────────────
shares = smoothed_shares_flat(EXPO_HIST)
EXPO_FUTURE = allocate_exposure(total_future, shares)
log(f"allocated exposure to {len(shares)} base nodes over {total_future.shape[0]} days")

# ══ SHARE-MODEL + ADDITIVE-COHERENCE VALIDATION ═════════════════════════════
_ok = True
_neg = (shares['share'] < -1e-12).sum()
log(f"(s1) shares non-negative: violations={_neg}  " + ("✓" if _neg == 0 else "❌")); _ok &= _neg == 0
_ssum = shares['share'].sum()
log(f"(s2) shares sum to 1: Σ={_ssum:.10f}  " + ("✓" if abs(_ssum-1) < 1e-6 else "❌")); _ok &= abs(_ssum-1) < 1e-6
_chk = (EXPO_FUTURE.groupby('forecast_date')['weight'].sum().reset_index()
        .merge(total_future, on='forecast_date'))
_adiff = (_chk['weight'] - _chk['total']).abs().max()
log(f"(s3) ADDITIVE coherence Σ_node exposure == parent total: maxdiff={_adiff:.2e}  "
    + ("✓" if _adiff < 1e-6 else "❌")); _ok &= _adiff < 1e-6
# no explosion/collapse over the horizon (vs recent history scale)
_recent_max = anchor[anchor['day'] >= anchor_last - pd.Timedelta(days=90)]['value'].max()
_expl = (_tot.max() > 5 * _recent_max) or (_tot.min() < 0)
log(f"(s4) horizon total within sane bounds (<=5× recent max {_recent_max:,.0f}): "
    + ("✓" if not _expl else "⚠️  check — possible explosion")); _ok &= not _expl
assert _ok, "EXPOSURE / SHARE VALIDATION FAILED — see (s1)–(s4)"
log("\n✅ EXPOSURE ADDITIVE TOP-DOWN VALIDATION PASSED "
    "(coherent time-varying horizon weights ready)")


HIERARCHY v1 (patch 2) — ADDITIVE EXPOSURE FORECAST
exposure history: 17,695 node-days, 52 base nodes
anchor exposure forecast: model=sarima, horizon=1100d, total range=[210,515, 604,258]
allocated exposure to 46 base nodes over 1100 days
(s1) shares non-negative: violations=0  ✓
(s2) shares sum to 1: Σ=1.0000000000  ✓
(s3) ADDITIVE coherence Σ_node exposure == parent total: maxdiff=0.00e+00  ✓
(s4) horizon total within sane bounds (<=5× recent max 287,627): ✓

✅ EXPOSURE ADDITIVE TOP-DOWN VALIDATION PASSED (coherent time-varying horizon weights ready)


In [79]:
# ══════════════════════════════════════════════════════════════════════════
# HIERARCHY v1 (patch 2) — ALL-METRIC BASE FORECASTING (single distributed job)
# One base-node series per (metric, base node); forecast via the UNCHANGED
# forecast_time_series_row executor path (no duplicate forecasting impl).
# ══════════════════════════════════════════════════════════════════════════
from functools import reduce
import json, time
from pyspark.sql import functions as F

log_section("HIERARCHY v1 (patch 2) — ALL-METRIC BASE FORECASTING")

HIER_METRICS = [m for m in METRICS if m in df.columns]
_absent = [m for m in METRICS if m not in df.columns]
if _absent:
    log(f"⚠️  metrics absent from source (skipped): {_absent}")
log(f"forecasting {len(HIER_METRICS)} metrics × {DIMS_LOOKUP.shape[0]} base nodes "
    f"= {len(HIER_METRICS) * DIMS_LOOKUP.shape[0]} base series")

# build one tasks DF spanning ALL metrics, then a single executor pass
def _tasks_for_metric(metric):
    return (build_base_fact(metric)
            .withColumn('group_key', F.concat_ws(BASE_SEP, *[F.col(c) for c in BASE_DIMS]))
            .groupBy('group_key')
            .agg(F.collect_list(F.struct('day', 'value')).alias('time_series_data'))
            .withColumn('metric', F.lit(metric))
            .withColumn('grouping_name', F.lit('__base__'))
            .select('metric', 'grouping_name', 'group_key', 'time_series_data'))

all_base_tasks = reduce(lambda a, b: a.unionByName(b),
                        [_tasks_for_metric(m) for m in HIER_METRICS])
_n = all_base_tasks.count()
log(f"created {_n} base-node tasks; running distributed forecast…")
_t = time.time()
base_results_all = [json.loads(r) for r in
                    all_base_tasks.repartition(min(_n, 400)).rdd
                    .map(forecast_time_series_row).collect()]
HIER_FORECAST_SECONDS = time.time() - _t
_done = sum(r.get('status') == 'completed' for r in base_results_all)
log(f"✓ base forecasting done in {HIER_FORECAST_SECONDS:.1f}s "
    f"({HIER_FORECAST_SECONDS/60:.1f} min)  completed={_done} "
    f"skipped={sum(r.get('status')=='skipped' for r in base_results_all)} "
    f"error={sum(r.get('status')=='error' for r in base_results_all)}")
assert _done > 0, "no base series completed — cannot roll up"


HIERARCHY v1 (patch 2) — ALL-METRIC BASE FORECASTING
forecasting 10 metrics × 52 base nodes = 520 base series
created 520 base-node tasks; running distributed forecast…
✓ base forecasting done in 24.7s (0.4 min)  completed=497 skipped=20 error=3


In [80]:
# ══════════════════════════════════════════════════════════════════════════
# HIERARCHY v1 (patch 2) — WEIGHTED ROLLUP → CANONICAL SCHEMA → PLOTS/EXPORTS
# Every (metric, reporting-cut) becomes a parsed_results-schema entry so the
# EXISTING plot_data_list builder + export logic consume it unchanged.
# APPROXIMATION: percentile rollups are exposure-weighted averages, NOT true
# quantile reconciliation.
# ══════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd
log_section("HIERARCHY v1 (patch 2) — ROLLUP, CANONICAL WIRING, EXPORTS")

# ── 1. roll every metric up to every reporting cut → canonical entries ──────
hier_parsed_results = []
for m in HIER_METRICS:
    util = is_util_metric(m)
    nfc, nhist, ntp = assemble_node_frames(base_results_all, m, DIMS_LOOKUP,
                                            EXPO_FUTURE, EXPO_HIST)
    if nfc.empty:
        log(f"  · {m}: no completed base nodes, skipped"); continue
    for cut, dims in REPORTING_CUTS.items():
        if any(d == 'product_segment' for d in dims) and not SEGMENT_MAP:
            continue
        hier_parsed_results += build_cut_entry(m, cut, dims, nfc, nhist, ntp, util)
log(f"built {len(hier_parsed_results)} canonical (metric × cut × group_key) entries")

# ── 2. wire into the canonical plot pipeline (ONE shared builder) ───────────
hier_plot_data_list = build_plot_data_list(hier_parsed_results)
log(f"build_plot_data_list → {len(hier_plot_data_list)} plot entries "
    "(schema identical to cell 19)")

# ── 3. tidy daily + monthly exports, derived from the SAME entries ──────────
_daily = []
for e in hier_parsed_results:
    r = e['results']['hierarchical_weighted']
    f = np.asarray(r['forecast'], float)
    lo = np.asarray(r['forecast_lower'], float); hi = np.asarray(r['forecast_upper'], float)
    last_hist = pd.to_datetime(list(e['train_dates']) + list(e['test_dates'])).max()
    dates = pd.date_range(last_hist + pd.Timedelta(days=1), periods=len(f), freq='D')
    _daily.append(pd.DataFrame({
        'metric': e['metric'], 'grouping': e['grouping'], 'group_key': e['group_key'],
        'model': e['best_model'], 'forecast_date': dates,
        'forecast_p50': f, 'forecast_p10': lo, 'forecast_p90': hi,
        'last_historical_date': last_hist, 'forecast_horizon_days': np.arange(1, len(f) + 1)}))
hier_daily_all = pd.concat(_daily, ignore_index=True)
hier_daily_all['year_month'] = hier_daily_all['forecast_date'].dt.to_period('M').astype(str)
hier_monthly_all = (hier_daily_all.groupby(['metric', 'grouping', 'group_key', 'model', 'year_month'])
                    .agg(avg_forecast_p50=('forecast_p50', 'mean'),
                         avg_forecast_p10=('forecast_p10', 'mean'),
                         avg_forecast_p90=('forecast_p90', 'mean'),
                         month_start_date=('forecast_date', 'min'),
                         month_end_date=('forecast_date', 'max'),
                         days_in_month_period=('forecast_date', 'nunique')).reset_index())
log(f"daily rows={len(hier_daily_all):,}  monthly rows={len(hier_monthly_all):,}  "
    f"metrics={hier_daily_all['metric'].nunique()}  cuts={hier_daily_all['grouping'].nunique()}")

# ══ VALIDATION across all metrics × cuts ════════════════════════════════════
_ok = True; _TOL = 1e-6
# (a) approximate rollup coherence per metric: All == region→All (weighted)
_worst = 0.0
for m in HIER_METRICS:
    nfc, _, _ = assemble_node_frames(base_results_all, m, DIMS_LOOKUP, EXPO_FUTURE, EXPO_HIST)
    if nfc.empty:
        continue
    ad = weighted_rollup(nfc, [], 'p50', 'forecast_date').set_index('forecast_date')['p50']
    reg = weighted_rollup(nfc, ['region_summary'], 'p50', 'forecast_date'); reg['wv'] = reg['p50']*reg['weight']
    av = (reg.groupby('forecast_date').agg(wv=('wv','sum'), w=('weight','sum'))
          .assign(p50=lambda x: x.wv/x.w)['p50'])
    _worst = max(_worst, float((ad - av).abs().max()))
log(f"(a) approx rollup coherence base→All == region→All (all metrics): "
    f"maxdiff={_worst:.2e}  " + ("✓" if _worst < _TOL else "❌")); _ok &= _worst < _TOL
# (b) intervals p10<=p50<=p90
_iv = ((hier_daily_all['forecast_p10'] > hier_daily_all['forecast_p50'] + 1e-9) |
       (hier_daily_all['forecast_p50'] > hier_daily_all['forecast_p90'] + 1e-9)).sum()
log(f"(b) interval ordering violations={_iv}  " + ("✓" if _iv == 0 else "❌")); _ok &= _iv == 0
# (c) bounded util metrics within [0,1]
_ub = hier_daily_all[hier_daily_all['metric'].apply(is_util_metric)]
_bad_b = ((_ub[['forecast_p10','forecast_p50','forecast_p90']] < -1e-9).any(axis=1) |
          (_ub[['forecast_p10','forecast_p50','forecast_p90']] > 1 + 1e-9).any(axis=1)).sum()
log(f"(c) util rows out of [0,1]={_bad_b}  " + ("✓" if _bad_b == 0 else "❌")); _ok &= _bad_b == 0
# (d) horizon length identical across cuts within each metric (not truncated)
_hz = hier_daily_all.groupby(['metric', 'grouping'])['forecast_date'].nunique()
_hz_ok = (_hz.groupby('metric').nunique() == 1).all()
log(f"(d) horizon length consistent across cuts per metric: " + ("✓" if _hz_ok else "❌")); _ok &= _hz_ok
# (e) monthly == mean(daily), recomputed independently
_chk = (hier_daily_all.groupby(['metric','grouping','group_key','year_month'])['forecast_p50']
        .mean().reset_index()
        .merge(hier_monthly_all, on=['metric','grouping','group_key','year_month']))
_mm = (_chk['forecast_p50'] - _chk['avg_forecast_p50']).abs().max()
log(f"(e) monthly == mean(daily): maxdiff={_mm:.2e}  " + ("✓" if _mm < _TOL else "❌")); _ok &= _mm < _TOL
assert _ok, "PATCH-2 VALIDATION FAILED — see (a)–(e)"
log("\n✅ PATCH-2 VALIDATION PASSED (all metrics × cuts): approximate coherence, "
    "intervals, bounds, horizon, monthly==mean(daily)")

# ── 4. persist hierarchy artifacts (distinct names; baseline exports intact) ─
try:
    save_df_to_s3('forecast_daily_values_hier', hier_daily_all)
    save_df_to_s3('forecast_monthly_values_hier', hier_monthly_all)
    log("✓ saved forecast_daily_values_hier / forecast_monthly_values_hier")
except Exception as _e:
    log(f"(export skipped: {str(_e)[:120]})")

# expose for downstream plotting cells if the user re-runs them against hierarchy:
#   plot_data_list = hier_plot_data_list
log("\nℹ️  To render plots/exports for the hierarchy instead of the baseline, set "
    "`plot_data_list = hier_plot_data_list` and re-run the plotting/export cells. "
    "Patch 3 will make this the default after the holdout backtest.")


HIERARCHY v1 (patch 2) — ROLLUP, CANONICAL WIRING, EXPORTS
built 490 canonical (metric × cut × group_key) entries
build_plot_data_list → 490 plot entries (schema identical to cell 19)
daily rows=585,660  monthly rows=19,820  metrics=10  cuts=7
(a) approx rollup coherence base→All == region→All (all metrics): maxdiff=7.45e-09  ✓
(b) interval ordering violations=0  ✓
(c) util rows out of [0,1]=0  ✓
(d) horizon length consistent across cuts per metric: ✓
(e) monthly == mean(daily): maxdiff=0.00e+00  ✓

✅ PATCH-2 VALIDATION PASSED (all metrics × cuts): approximate coherence, intervals, bounds, horizon, monthly==mean(daily)
  → s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/20260723_221127/forecast_daily_values_hier.csv
  → s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/20260723_221127/forecast_daily_values_hier.xlsx
  → s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/20260723_221127/forecast_monthly_values_hier.csv
  → s3://jbok-sandbox-test/jbok/time-series-fc

# 🌳🌳🌳 Hierarchical Forecasting (v1 — Patch 3: backtest, validation, cleanup)

Patch 3 closes out the work:

1. **Holdout backtest — hierarchy vs the non-hierarchical baseline.** On a shared
   holdout window we run *both* estimators on truncated history (day ≤ cutoff)
   in a **single** distributed job, then score each against actuals on the
   **common planning target** (the exposure-weighted actual aggregate). Reports
   per-cut **accuracy** (MAE/RMSE/MAPE/bias), **stability** (path volatility),
   and the **monthly planning delta** — plus **wall-clock runtime** so we can
   confirm it stays operationally acceptable on SparkCaster.
2. **Export & notebook consistency validation.** Canonical schema preserved end
   to end; `build_plot_data_list` renders every cut; monthly outputs equal
   `mean(daily)`; hierarchy daily/monthly frames internally consistent.
3. **Cleanup / restart-run-all.** An idempotent `HIER_AS_DEFAULT` switch points
   `plot_data_list` at the hierarchy so the existing plot/export cells render
   the hierarchy when re-run, without reordering or duplicating logic. Guards
   make the section safe under restart-and-run-all.

The backtest defaults to a **representative metric subset and key cuts** to keep
runtime bounded (spec: prove one metric end-to-end before scaling); widen
`BT_METRICS` / `BT_CUTS` to cover everything once runtime is confirmed.


In [81]:
# ══════════════════════════════════════════════════════════════════════════
# HIERARCHY v1 (patch 3) — HOLDOUT BACKTEST: hierarchy vs non-hier baseline
# Both estimators forecast truncated history (day <= cutoff) in ONE Spark job;
# scored on the common planning target = exposure-weighted actual aggregate.
# ══════════════════════════════════════════════════════════════════════════
from functools import reduce
import json, time
import numpy as np, pandas as pd
from pyspark.sql import functions as F

log_section("HIERARCHY v1 (patch 3) — HOLDOUT BACKTEST")

# ── config (widen once runtime is confirmed) ────────────────────────────────
BT_HOLDOUT_DAYS = 60
# representative subset — dedupe so a fallback pick can't repeat a metric
BT_METRICS = list(dict.fromkeys(
    m for m in (['gpu_util_p50', 'chip_power_fleet_p50'] + HIER_METRICS) if m in df.columns))[:2]
BT_CUTS = ['All', 'region_summary', 'product_resolved', 'customer_segment']
_maxday = df.select(F.max('day')).first()[0]
BT_CUTOFF = pd.Timestamp(_maxday) - pd.Timedelta(days=BT_HOLDOUT_DAYS)
log(f"cutoff={BT_CUTOFF.date()}  holdout={BT_HOLDOUT_DAYS}d  "
    f"metrics={BT_METRICS}  cuts={BT_CUTS}")

# ── error helpers (offline unit-tested in backtest_core) ────────────────────
def eval_errors(pred_by_date, actual_by_date):
    j = pd.concat([pred_by_date.rename('pred'), actual_by_date.rename('act')], axis=1).dropna()
    if len(j) == 0:
        return {'n': 0, 'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan, 'bias': np.nan}
    err = j['pred'] - j['act']; denom = j['act'].where(j['act'] != 0, np.nan)
    return {'n': int(len(j)), 'MAE': float(err.abs().mean()),
            'RMSE': float(np.sqrt((err ** 2).mean())),
            'MAPE': float((err.abs() / denom).mean() * 100), 'bias': float(err.mean())}
def _stab(p):
    p = p.sort_index().to_numpy(float)
    return float(np.mean(np.abs(np.diff(p)))) if len(p) > 1 else 0.0

_cut_dims = {c: REPORTING_CUTS[c] for c in BT_CUTS}

# ── build ONE tasks DF: hierarchy base nodes + baseline cuts (truncated) ────
def _hier_tasks(metric):
    bf = build_base_fact(metric).filter(F.col('day') <= F.lit(BT_CUTOFF))
    return (bf.withColumn('group_key', F.concat_ws(BASE_SEP, *[F.col(c) for c in BASE_DIMS]))
            .groupBy('group_key').agg(F.collect_list(F.struct('day', 'value')).alias('time_series_data'))
            .withColumn('metric', F.lit(metric)).withColumn('grouping_name', F.lit('__bt_base__'))
            .select('metric', 'grouping_name', 'group_key', 'time_series_data'))

def _baseline_tasks(metric, cut):
    dims = _cut_dims[cut]
    src = df.withColumn('product_segment', _segment_col()).filter(F.col('day') <= F.lit(BT_CUTOFF))
    if dims:
        agg = (src.groupBy(*dims, 'day').agg(F.avg(F.col(metric)).alias('value'))
               .withColumn('group_key', F.concat_ws('|', *[F.col(d) for d in dims])))
    else:
        agg = (src.groupBy('day').agg(F.avg(F.col(metric)).alias('value'))
               .withColumn('group_key', F.lit('All')))
    return (agg.groupBy('group_key').agg(F.collect_list(F.struct('day', 'value')).alias('time_series_data'))
            .withColumn('metric', F.lit(metric)).withColumn('grouping_name', F.lit(f'baseline:{cut}'))
            .select('metric', 'grouping_name', 'group_key', 'time_series_data'))

_task_frames = []
for m in BT_METRICS:
    _task_frames.append(_hier_tasks(m))
    for c in BT_CUTS:
        _task_frames.append(_baseline_tasks(m, c))
bt_tasks = reduce(lambda a, b: a.unionByName(b), _task_frames)
_n = bt_tasks.count()
log(f"backtest tasks: {_n} (hier base + baseline cuts, truncated). Forecasting…")
_t = time.time()
bt_results = [json.loads(r) for r in
              bt_tasks.repartition(min(_n, 400)).rdd.map(forecast_time_series_row).collect()]
BT_SECONDS = time.time() - _t
log(f"✓ backtest forecasting: {BT_SECONDS:.1f}s ({BT_SECONDS/60:.1f} min) for "
    f"{sum(r.get('status')=='completed' for r in bt_results)}/{_n} series")

# ── actuals in the holdout window (base grain + exposure), pulled once ──────
_hold = (df.withColumn('_expo', _exposure_col()).withColumn('product_segment', _segment_col())
         .withColumn('group_key', F.concat_ws(BASE_SEP, *[F.col(c) for c in BASE_DIMS]))
         .filter((F.col('day') > F.lit(BT_CUTOFF)) & (F.col('day') <= F.lit(_maxday))))
_hold_cols = list(dict.fromkeys(  # dedupe → no duplicate pandas columns
    BASE_DIMS + ['product_segment', 'group_key', 'day', '_expo'] + BT_METRICS))
HOLD = _hold.select(*_hold_cols).toPandas()
HOLD = HOLD.loc[:, ~HOLD.columns.duplicated()].copy()   # belt-and-suspenders
HOLD['day'] = pd.to_datetime(HOLD['day'])

def _weighted_actual(metric, cut):
    dims = _cut_dims[cut]
    d = HOLD.copy()
    # position-based math avoids index/column-alignment surprises
    d['w'] = np.asarray(d['_expo'], float).clip(min=0)
    d['wv'] = np.asarray(d[metric], float) * d['w'].to_numpy()
    gb = (dims + ['day']) if dims else ['day']
    g = d.groupby(gb, dropna=False).agg(wv=('wv', 'sum'), w=('w', 'sum'),
                                        mean_v=(metric, 'mean')).reset_index()
    g['val'] = np.where(g['w'] > 0, g['wv'] / g['w'], g['mean_v'])
    g['group_key'] = (g[dims].astype(str).agg('|'.join, axis=1) if dims else 'All')
    return g[['group_key', 'day', 'val']]

def _unweighted_actual(metric, cut):
    dims = _cut_dims[cut]
    gb = (dims + ['day']) if dims else ['day']
    g = HOLD.groupby(gb, dropna=False).agg(val=(metric, 'mean')).reset_index()
    g['group_key'] = (g[dims].astype(str).agg('|'.join, axis=1) if dims else 'All')
    return g[['group_key', 'day', 'val']]

# ── hierarchy holdout forecast per cut (roll up base forecasts, actual expo) ─
def _hier_holdout(metric):
    rows = []
    for r in bt_results:
        if r.get('grouping_name') != '__bt_base__' or r.get('metric') != metric \
                or r.get('status') != 'completed' or not r.get('best_model'):
            continue
        f = np.asarray(r['results'][r['best_model']].get('forecast', []), float)[:BT_HOLDOUT_DAYS]
        if len(f) == 0:
            continue
        dates = pd.date_range(BT_CUTOFF + pd.Timedelta(days=1), periods=len(f), freq='D')
        rows.append(pd.DataFrame({'group_key': r['group_key'], 'day': dates, 'p50': f}))
    if not rows:
        return {}
    nfc = pd.concat(rows, ignore_index=True).merge(
        DIMS_LOOKUP.reset_index(), on='group_key', how='left')
    # actual holdout exposure as the (known-composition) rollup weight
    w = HOLD.groupby(['group_key', 'day'])['_expo'].sum().clip(lower=0).rename('weight').reset_index()
    nfc = nfc.merge(w, on=['group_key', 'day'], how='left'); nfc['weight'] = nfc['weight'].fillna(0.0)
    out = {}
    for cut in BT_CUTS:
        dims = _cut_dims[cut]
        roll = weighted_rollup(nfc, dims, 'p50', 'day')
        roll['group_key'] = (roll[dims].astype(str).agg('|'.join, axis=1) if dims else 'All')
        out[cut] = roll[['group_key', 'day', 'p50']]
    return out

def _baseline_holdout(metric, cut):
    out = {}
    for r in bt_results:
        if r.get('grouping_name') != f'baseline:{cut}' or r.get('metric') != metric \
                or r.get('status') != 'completed' or not r.get('best_model'):
            continue
        f = np.asarray(r['results'][r['best_model']].get('forecast', []), float)[:BT_HOLDOUT_DAYS]
        dates = pd.date_range(BT_CUTOFF + pd.Timedelta(days=1), periods=len(f), freq='D')
        out[r['group_key']] = pd.Series(f, index=dates)
    return out

# ── score both methods on the common planning target ───────────────────────
records = []
for m in BT_METRICS:
    util = is_util_metric(m)
    wact = {c: _weighted_actual(m, c) for c in BT_CUTS}
    uact = {c: _unweighted_actual(m, c) for c in BT_CUTS}
    hh = _hier_holdout(m)
    for cut in BT_CUTS:
        bl = _baseline_holdout(m, cut)
        wa = wact[cut]; ua = uact[cut]
        for gk in wa['group_key'].unique():
            wser = wa[wa['group_key'] == gk].set_index('day')['val']
            user = ua[ua['group_key'] == gk].set_index('day')['val'] if gk in set(ua['group_key']) else wser
            # hierarchy forecast for this cut/group
            hcut = hh.get(cut)
            hser = (hcut[hcut['group_key'] == gk].set_index('day')['p50']
                    if hcut is not None and gk in set(hcut['group_key']) else pd.Series(dtype=float))
            bser = bl.get(gk, pd.Series(dtype=float))
            if len(hser):
                records.append({'metric': m, 'cut': cut, 'group_key': gk, 'method': 'hierarchy',
                                'target': 'weighted_actual', **eval_errors(hser, wser), 'stability': _stab(hser)})
            if len(bser):
                records.append({'metric': m, 'cut': cut, 'group_key': gk, 'method': 'baseline',
                                'target': 'weighted_actual', **eval_errors(bser, wser), 'stability': _stab(bser)})
                records.append({'metric': m, 'cut': cut, 'group_key': gk, 'method': 'baseline',
                                'target': 'unweighted_actual', **eval_errors(bser, user), 'stability': _stab(bser)})

bt_detail = pd.DataFrame(records)
_common = bt_detail[bt_detail['target'] == 'weighted_actual']
bt_summary = (_common.pivot_table(index=['metric', 'cut'], columns='method', values='MAE')
              .rename(columns={'baseline': 'MAE_baseline', 'hierarchy': 'MAE_hierarchy'}))
if {'MAE_baseline', 'MAE_hierarchy'} <= set(bt_summary.columns):
    bt_summary['pct_improvement'] = (100 * (bt_summary['MAE_baseline'] - bt_summary['MAE_hierarchy'])
                                     / bt_summary['MAE_baseline'].replace(0, np.nan))
bt_summary = bt_summary.reset_index()

# ── monthly planning delta at key cuts (hierarchy vs baseline, holdout) ─────
def _monthly(series):
    return series.groupby(series.index.to_period('M')).mean()
_mrows = []
for m in BT_METRICS:
    hh = _hier_holdout(m); bl_all = {c: _baseline_holdout(m, c) for c in BT_CUTS}
    for cut in BT_CUTS:
        hc = hh.get(cut)
        if hc is None:
            continue
        for gk in hc['group_key'].unique():
            hser = hc[hc['group_key'] == gk].set_index('day')['p50']
            bser = bl_all[cut].get(gk, pd.Series(dtype=float))
            if len(bser) == 0:
                continue
            hm, bm = _monthly(hser), _monthly(bser)
            for per in hm.index.intersection(bm.index):
                _mrows.append({'metric': m, 'cut': cut, 'group_key': gk, 'year_month': str(per),
                               'hier_monthly': float(hm[per]), 'baseline_monthly': float(bm[per]),
                               'delta': float(hm[per] - bm[per])})
bt_monthly_delta = pd.DataFrame(_mrows)

log("\n── Backtest accuracy on common planning target (weighted actual) ──")
log(bt_summary.to_string(index=False))
if not bt_monthly_delta.empty:
    log("\n── Monthly planning delta (hierarchy − baseline), sample ──")
    log(bt_monthly_delta.head(12).to_string(index=False))
_impr = bt_summary['pct_improvement'].dropna() if 'pct_improvement' in bt_summary else pd.Series(dtype=float)
log(f"\nRUNTIME: {BT_SECONDS:.1f}s for {len(BT_METRICS)} metrics × {len(BT_CUTS)} cuts "
    f"({'OK' if BT_SECONDS < 1800 else '⚠️ review'} vs 30-min budget)")
if len(_impr):
    log(f"Hierarchy vs baseline MAE on planning target: mean {_impr.mean():+.1f}% "
        f"(positive = hierarchy closer to the coherent target). "
        "NOTE: baseline's unweighted average is a different estimator of the same "
        "aggregate; this quantifies the coherence gap, not just raw skill.")
try:
    save_df_to_s3('backtest_summary_hier_vs_baseline', bt_summary)
    save_df_to_s3('backtest_detail_hier_vs_baseline', bt_detail)
    log("✓ saved backtest_summary / backtest_detail")
except Exception as _e:
    log(f"(export skipped: {str(_e)[:120]})")


HIERARCHY v1 (patch 3) — HOLDOUT BACKTEST
cutoff=2026-04-24  holdout=60d  metrics=['gpu_util_p50', 'chip_power_fleet_p50']  cuts=['All', 'region_summary', 'product_resolved', 'customer_segment']
backtest tasks: 140 (hier base + baseline cuts, truncated). Forecasting…
✓ backtest forecasting: 8.1s (0.1 min) for 134/140 series

── Backtest accuracy on common planning target (weighted actual) ──
              metric              cut  MAE_baseline  MAE_hierarchy  pct_improvement
chip_power_fleet_p50              All  3.324368e+06   1.227143e+06        63.086433
chip_power_fleet_p50 customer_segment  2.476902e+06   1.219136e+06        50.779805
chip_power_fleet_p50 product_resolved  1.369504e+06   8.196795e+05        40.147708
chip_power_fleet_p50   region_summary  2.247318e+06   7.392169e+05        67.106703
        gpu_util_p50              All  2.110706e-01   7.675329e-02        63.636207
        gpu_util_p50 customer_segment  2.346344e-01   8.351962e-02        64.404360
        gpu_util_

In [82]:
# ══════════════════════════════════════════════════════════════════════════
# HIERARCHY v1 (patch 3) — EXPORT/NOTEBOOK CONSISTENCY + RETROFIT + CLEANUP
# ══════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd
log_section("HIERARCHY v1 (patch 3) — CONSISTENCY VALIDATION & CLEANUP")

_ok = True
_CANON = {'metric', 'grouping', 'group_key', 'status', 'best_model', 'results',
          'train_dates', 'test_dates', 'train_values', 'test_values'}

# (1) canonical schema preserved end-to-end: baseline vs hierarchy entries share
#     the same contract (so plotting/export are agnostic to which produced them)
_base_done = next((r for r in all_results if r.get('status') == 'completed'), None)
_hier_done = next((r for r in hier_parsed_results if r.get('status') == 'completed'), None)
assert _hier_done is not None, "no completed hierarchy entries"
_missing = _CANON - set(_hier_done)
log(f"(1) hierarchy entry has canonical keys: " + ("✓" if not _missing else f"❌ missing {_missing}"))
_ok &= not _missing
if _base_done is not None:
    _delta = _CANON - set(_base_done)
    log(f"    baseline entry canonical keys: " + ("✓ same contract" if not _delta else f"⚠️ {_delta}"))

# (2) plotting contract: build_plot_data_list works and export-critical fields exist
_pdl = build_plot_data_list(hier_parsed_results)
assert len(_pdl) > 0, "build_plot_data_list produced nothing"
_e = _pdl[0]; _r = _e['results'][_e['best_model']]
_md_ok = hasattr(_r['metadata']['train_dates'], 'max')                 # cell-27 needs .max()
_len_ok = len(_r['forecast']) == len(_r['forecast_lower']) == len(_r['forecast_upper'])
_iv_ok = bool(((_r['forecast_lower'] <= _r['forecast'] + 1e-9) &
               (_r['forecast'] <= _r['forecast_upper'] + 1e-9)).all())
log(f"(2) plot/export contract: entries={len(_pdl)} metadata.max()={_md_ok} "
    f"interval-aligned={_len_ok} ordered={_iv_ok}  "
    + ("✓" if (_md_ok and _len_ok and _iv_ok) else "❌"))
_ok &= _md_ok and _len_ok and _iv_ok

# (3) monthly outputs internally consistent with daily hierarchy (recompute)
_chk = (hier_daily_all.groupby(['metric', 'grouping', 'group_key', 'year_month'])['forecast_p50']
        .mean().reset_index()
        .merge(hier_monthly_all, on=['metric', 'grouping', 'group_key', 'year_month']))
_mm = float((_chk['forecast_p50'] - _chk['avg_forecast_p50']).abs().max())
log(f"(3) monthly == mean(daily): maxdiff={_mm:.2e}  " + ("✓" if _mm < 1e-6 else "❌")); _ok &= _mm < 1e-6

# (4) required export columns present (no silently-missing fields downstream)
_need_daily = {'metric', 'grouping', 'group_key', 'model', 'forecast_date',
               'forecast_p10', 'forecast_p50', 'forecast_p90'}
_miss_daily = _need_daily - set(hier_daily_all.columns)
log(f"(4) daily export columns present: " + ("✓" if not _miss_daily else f"❌ {_miss_daily}"))
_ok &= not _miss_daily

assert _ok, "CONSISTENCY VALIDATION FAILED — see (1)–(4)"
log("✅ CONSISTENCY VALIDATION PASSED (canonical schema + plots + monthly + exports)")

# ── RETROFIT / CLEANUP: opt-in switch to make the hierarchy the default output ─
# Idempotent + restart-run-all safe: only rebinds when hierarchy objects exist.
HIER_AS_DEFAULT = True   # set False to keep the baseline as the plotted/exported result
if HIER_AS_DEFAULT and 'hier_plot_data_list' in dir() and hier_plot_data_list:
    plot_data_list = hier_plot_data_list          # existing plot/export cells now render hierarchy
    all_plots = hier_plot_data_list
    # canonical daily frame in the exact schema cell 31 consumes (forecast_value alias kept)
    df_all_forecasts = hier_daily_all.assign(forecast_value=hier_daily_all['forecast_p50'])
    log(f"🔀 HIER_AS_DEFAULT: plot_data_list ← hierarchy ({len(plot_data_list)} series); "
        "re-run the plotting/export cells above to emit hierarchy artifacts.")
else:
    log("HIER_AS_DEFAULT off — baseline remains the plotted/exported result.")

log("\nRESTART-AND-RUN-ALL: this section is additive and idempotent — it defines "
    "its own functions, reuses df/spark/forecast_time_series_row/METRICS from "
    "above, and rebinds only when hierarchy objects exist. No manual cell order "
    "or stale state is required.")
log_section("✅ HIERARCHY v1 COMPLETE (patches 1–3): structure, data, shares, "
            "approximate coherence, intervals, backtest, exports all validated")


HIERARCHY v1 (patch 3) — CONSISTENCY VALIDATION & CLEANUP
(1) hierarchy entry has canonical keys: ✓
    baseline entry canonical keys: ✓ same contract
(2) plot/export contract: entries=490 metadata.max()=True interval-aligned=True ordered=True  ✓
(3) monthly == mean(daily): maxdiff=0.00e+00  ✓
(4) daily export columns present: ✓
✅ CONSISTENCY VALIDATION PASSED (canonical schema + plots + monthly + exports)
🔀 HIER_AS_DEFAULT: plot_data_list ← hierarchy (490 series); re-run the plotting/export cells above to emit hierarchy artifacts.

RESTART-AND-RUN-ALL: this section is additive and idempotent — it defines its own functions, reuses df/spark/forecast_time_series_row/METRICS from above, and rebinds only when hierarchy objects exist. No manual cell order or stale state is required.
✅ HIERARCHY v1 COMPLETE (patches 1–3): structure, data, shares, approximate coherence, intervals, backtest, exports all validated
